In [ ]:
import os
import cdsapi
from scipy.signal import correlate
import pandas as pd
import shutil
import numpy as np
import pyproj
import pygrib
import xarray as xr
import s3fs
import dataretrieval.nwis as nwis
from google.cloud import storage
from datetime import datetime, timedelta, date
import json
from haversine import haversine, Unit
import warnings
import geopandas as gpd
from shapely.geometry import Point
import resource
from pyproj import Transformer
import pickle
from collections import defaultdict
import subprocess
from pathlib import Path
import requests
from pytz import utc
import matplotlib.pyplot as plt
from pandas import Timestamp
import matplotlib.dates as mdates
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import botocore
from math import sqrt
from concurrent.futures import ThreadPoolExecutor
import fsspec
from IPython.display import Image, display
import tempfile


# Constants
CFSToCMS_CONVERSION_FACTOR = 0.0283168466

# Define time range
start_usgs = '1979-01-01'

today = datetime.today()
# Format it as YYYY-MM-DD
end_usgs = today.strftime('%Y-%m-%d')

# Convert end date to datetime object and get start date for forecast
end_usgs_datetime = datetime.strptime(end_usgs, '%Y-%m-%d')
start_forecast_datetime = end_usgs_datetime - timedelta(days=1)
start_forecast = start_forecast_datetime.strftime('%Y-%m-%d')

# Specify the USGS site code
site_code = '11160500'

# Fetch daily data for the site
df = nwis.get_record(sites=site_code, service='dv', parameterCd='00060', statCd='00003',
                     start=start_usgs, end=end_usgs)

# Log-transform the flow data; we add 1 to handle cases where the value is 0
df['log_discharge'] = np.log(df['00060_Mean'].astype(float) + 1)
# Keep only the relevant column
df = df[['log_discharge']]
# Reverse the log transformation to get back the discharge in cfs
df['discharge_cfs'] = np.exp(df['log_discharge']) - 1
# Convert discharge from cfs to cms
df['discharge_cms'] = df['discharge_cfs'] * CFSToCMS_CONVERSION_FACTOR
# Optionally, log-transform the discharge in cms
df['log_discharge_cms'] = np.log(df['discharge_cms'] + 1)


# Fetch metadata for the USGS site without specifying fields
site_info = nwis.get_record(sites=site_code, service='site')
station_name = site_info['station_nm'][0] 

# Extract latitude and longitude
latitude = float(site_info['dec_lat_va'][0])
longitude = float(site_info['dec_long_va'][0])

# Combine into a single target_location tuple
target_location = (latitude, longitude)
target_lat, target_lon = target_location
print(f"The coordinates for site {site_code} are {target_location}")

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import os
import fsspec

# Define the period, location of interest, and directory
lat, lon = target_lat, target_lon  # Make sure to define target_lat and target_lon
dir = "/data/muscat_data/jaguir26/project1_ucsc_phd"
site_code = "11160500"

# Define data access details for both datasets
datasets = {
    "new": {
        "start_date": "1979-01-01",
        "end_date": "2023-12-31",
        "zarr_path": "s3://noaa-nwm-retrospective-3-0-pds/CONUS/zarr/chrtout.zarr",
        "csv_file": os.path.join(dir, f'{site_code}_nws_retro.csv')
    },
    "old": {
        "start_date": "1979-01-01",
        "end_date": "2020-12-31",
        "zarr_path": "s3://noaa-nwm-retrospective-2-1-zarr-pds/chrtout.zarr",
        "csv_file": os.path.join(dir, f'{site_code}_nws_retro_old.csv')
    }
}

for key, details in datasets.items():
    # Check if the file already exists
    if os.path.exists(details["csv_file"]):
        print(f"File already exists: {details['csv_file']}")
    else:
        # Open the dataset from Zarr format on AWS
        ds = xr.open_zarr(fsspec.get_mapper(details["zarr_path"], anon=True), consolidated=True)

        # Create a DataFrame for feature_id, latitude, and longitude
        feature_locations = pd.DataFrame({
            'feature_id': ds['feature_id'].values,
            'latitude': ds['latitude'].values,
            'longitude': ds['longitude'].values
        })

        # Calculate the Euclidean distance from each feature to the specified lat/lon
        distances = np.sqrt((feature_locations['latitude'] - lat)**2 + (feature_locations['longitude'] - lon)**2)
        nearest_feature_id = feature_locations.loc[distances.idxmin(), 'feature_id']

        # Subset for the specific years and nearest feature_id
        subset = ds.sel(time=slice(details["start_date"], details["end_date"]), feature_id=nearest_feature_id)

        # Process the streamflow data
        streamflow = subset['streamflow']
        streamflow_data = streamflow.compute()

        # Convert xarray DataArray to pandas DataFrame
        streamflow_df = streamflow_data.to_dataframe().reset_index()

        # Extract date from index and add as a column
        streamflow_df['Date'] = streamflow_df['time']
        streamflow_df = streamflow_df.drop('time', axis=1)

        # Save the DataFrame as a CSV file
        streamflow_df.to_csv(details["csv_file"], index=False)
        print(f"Data saved as CSV at {details['csv_file']}")

print("Data processing completed for both new and old datasets.")


In [ ]:
df_usgs = df
df_usgs['Date'] = df_usgs.index

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Define file paths
dir = "/data/muscat_data/jaguir26/project1_ucsc_phd"
project_input_dir = "/data/muscat_data/jaguir26/projects/Project/Input/Retrospective_Analysis/GLOFAS"
site_code = "11160500"
nwm_new_path = os.path.join(dir, f'{site_code}_nws_retro.csv')
nwm_old_path = os.path.join(dir, f'{site_code}_nws_retro_old.csv')
glofas_path = os.path.join(project_input_dir, 'glofas_1979_2023', 'glofas_streamflow_data.csv')
df_glofas = pd.read_csv(glofas_path)


# Load DataFrames
df_nwm_new = pd.read_csv(nwm_new_path)
df_nwm_old = pd.read_csv(nwm_old_path)
df_glofas = pd.read_csv(glofas_path)


# Convert 'Date' to datetime format and set as index, ensuring timezone-naive
df_nwm_new['Date'] = pd.to_datetime(df_nwm_new['Date']).dt.tz_localize(None)
df_nwm_old['Date'] = pd.to_datetime(df_nwm_old['Date']).dt.tz_localize(None)
df_glofas['Date'] = pd.to_datetime(df_glofas['Date']).dt.tz_localize(None)
df_usgs['Date'] = pd.to_datetime(df_usgs['Date']).dt.tz_localize(None)  # Assuming this step is done similarly

df_nwm_new.set_index('Date', inplace=True)
df_nwm_old.set_index('Date', inplace=True)
df_glofas.set_index('Date', inplace=True)
df_usgs.set_index('Date', inplace=True)  # Assuming this DataFrame is similarly processed


# Assuming df_usgs is already loaded and has 'Date' as datetime index
# Start and end dates
start_date = '2018-01-01'
end_date = '2023-02-01'

# Filter the data between the specified dates and create copies to avoid SettingWithCopyWarning
filtered_nwm_new = df_nwm_new.loc[start_date:end_date].copy()
filtered_nwm_old = df_nwm_old.loc[start_date:end_date].copy()
filtered_usgs = df_usgs.loc[start_date:end_date].copy()
filtered_glofas = df_glofas.loc[start_date:end_date].copy()


# Transform the data by adding 1 and then taking the logarithm
filtered_nwm_new['log_streamflow'] = np.log1p(filtered_nwm_new['streamflow'])
filtered_nwm_old['log_streamflow'] = np.log1p(filtered_nwm_old['streamflow'])
filtered_usgs['log_discharge_cms'] = np.log1p(filtered_usgs['discharge_cms'])
filtered_glofas['log_streamflow'] = np.log1p(filtered_glofas['Streamflow'])

# Standardize the data
# filtered_nwm_new['std_streamflow'] = (filtered_nwm_new['log_streamflow'] - filtered_nwm_new['log_streamflow'].mean()) / filtered_nwm_new['log_streamflow'].std()
# filtered_nwm_old['std_streamflow'] = (filtered_nwm_old['log_streamflow'] - filtered_nwm_old['log_streamflow'].mean()) / filtered_nwm_old['log_streamflow'].std()
# filtered_usgs['std_discharge_cms'] = (filtered_usgs['log_discharge_cms'] - filtered_usgs['log_discharge_cms'].mean()) / filtered_usgs['log_discharge_cms'].std()
# filtered_glofas['std_streamflow'] = (filtered_glofas['log_streamflow'] - filtered_glofas['log_streamflow'].mean()) / filtered_glofas['log_streamflow'].std()


# Plotting the transformed data
plt.figure(figsize=(20, 7))
plt.plot(filtered_nwm_old.index, filtered_nwm_old['log_streamflow'], label='log(NWM2.1 + 1)', color='pink')
plt.plot(filtered_usgs.index, filtered_usgs['log_discharge_cms'], label='log(USGS + 1)', linestyle = 'dashed', color='green', markersize = 1)
plt.xlabel('Date')
plt.ylabel('NOT st - log(Discharge + 1)')
plt.title('NWM-Retrospective vs USGS (2018-2023)')
plt.legend()
plt.grid(True)
plt.show()


# Plotting the transformed data
plt.figure(figsize=(20, 7))
plt.plot(filtered_nwm_new.index, filtered_nwm_new['log_streamflow'], label='log(NWM3.0 + 1)', color='darkred')
plt.plot(filtered_usgs.index, filtered_usgs['log_discharge_cms'], label='log(USGS + 1)', linestyle = 'dashed', color='green', markersize = 1)
plt.xlabel('Date')
plt.ylabel('NOT st - log(Discharge + 1)')
plt.title('NWM-Retrospective vs USGS (2018-2023)')
plt.legend()
plt.grid(True)
plt.show()

# Plotting the transformed data
plt.figure(figsize=(20, 7))
plt.plot(filtered_nwm_new.index, filtered_nwm_new['log_streamflow'], label='log(NWM3.0 + 1)', color='darkred')
plt.plot(filtered_nwm_old.index, filtered_nwm_old['log_streamflow'], label='log(NWM2.1 + 1)', color='pink')
plt.plot(filtered_usgs.index, filtered_usgs['log_discharge_cms'], label='log(USGS + 1)', linestyle = 'dashed', color='green', markersize = 1)
plt.xlabel('Date')
plt.ylabel('NOT st - log(Discharge + 1)')
plt.title('NWM-Retrospective vs USGS (2018-2023)')
plt.legend()
plt.grid(True)
plt.show()

# Plotting the transformed data
plt.figure(figsize=(20, 7))
plt.plot(filtered_nwm_new.index, filtered_nwm_new['log_streamflow'], label='log(NWM3.0 + 1)', color='darkred')
plt.plot(filtered_nwm_old.index, filtered_nwm_old['log_streamflow'], label='log(NWM2.1 + 1)', color='pink')
plt.plot(filtered_usgs.index, filtered_usgs['log_discharge_cms'], label='log(USGS + 1)', linestyle = 'dashed', color='green', markersize = 1)
plt.xlabel('Date')
plt.ylabel('NOT st - log(Discharge + 1)')
plt.title('NWM-Retrospective vs USGS (2018-2023)')
plt.legend()
plt.grid(True)
plt.show()


# Plotting the transformed data
plt.figure(figsize=(20, 7))
plt.plot(filtered_nwm_new.index, filtered_nwm_new['log_streamflow'], label='log(NWM3.0 + 1)', color='darkred')
plt.plot(filtered_usgs.index, filtered_usgs['log_discharge_cms'], label='log(USGS + 1)', linestyle = 'dashed', color='green', markersize = 1)
plt.plot(filtered_glofas.index, filtered_glofas['log_streamflow'], label='log(GloFAS + 1)', color='orange')
plt.xlabel('Date')
plt.ylabel('NOT st - log(Discharge + 1)')
plt.title('NWM-Retrospective vs GloFAS vs USGS (2018-2023)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:

# Assuming df_usgs is already loaded and has 'Date' as datetime index
# Start and end dates
start_date = '2018-01-01'
end_date = '2023-02-01'

filtered_nwm_new = df_nwm_new.loc[start_date:end_date].copy()
filtered_usgs = df_usgs.loc[start_date:end_date].copy()
filtered_glofas = df_glofas.loc[start_date:end_date].copy()


# Transform the data by adding 1 and then taking the logarithm
filtered_nwm_new['log_streamflow'] = np.log1p(filtered_nwm_new['streamflow'])
filtered_usgs['log_discharge_cms'] = np.log1p(filtered_usgs['discharge_cms'])
filtered_glofas['log_streamflow'] = np.log1p(filtered_glofas['Streamflow'])

# # Standardize the data
# filtered_nwm_new['std_streamflow'] = (filtered_nwm_new['log_streamflow'] - filtered_nwm_new['log_streamflow'].mean()) / filtered_nwm_new['log_streamflow'].std()
# filtered_usgs['std_discharge_cms'] = (filtered_usgs['log_discharge_cms'] - filtered_usgs['log_discharge_cms'].mean()) / filtered_usgs['log_discharge_cms'].std()
# filtered_glofas['std_streamflow'] = (filtered_glofas['log_streamflow'] - filtered_glofas['log_streamflow'].mean()) / filtered_glofas['log_streamflow'].std()

# # Resample data to daily frequency by averaging hourly values
# daily_nwm_new = filtered_nwm_new['std_streamflow'].resample('D').mean()
# daily_usgs = filtered_usgs['std_discharge_cms'].resample('D').mean()
# daily_glofas = filtered_glofas['std_streamflow'].resample('D').mean()

# Resample data to daily frequency by averaging hourly values
daily_nwm_new = filtered_nwm_new['log_streamflow'].resample('D').mean()
daily_usgs = filtered_usgs['log_discharge_cms'].resample('D').mean()
daily_glofas = filtered_glofas['log_streamflow'].resample('D').mean()

# Convert all datetime indices to timezone-naive
daily_nwm_new.index = daily_nwm_new.index.tz_localize(None)
daily_usgs.index = daily_usgs.index.tz_localize(None)
daily_glofas.index = daily_glofas.index.tz_localize(None)

# Find the maximum start date and the minimum end date among the datasets
start_date = max(daily_nwm_new.index.min(), daily_usgs.index.min(), daily_glofas.index.min())
end_date = min(daily_nwm_new.index.max(), daily_usgs.index.max(), daily_glofas.index.max())

# Align all datasets to the common date range
common_daily_nwm_new = daily_nwm_new.loc[start_date:end_date]
common_daily_usgs = daily_usgs.loc[start_date:end_date]
common_daily_glofas = daily_glofas.loc[start_date:end_date]

# Create a new DataFrame combining all time series
combined_data = pd.DataFrame({
    'NWS3.0': common_daily_nwm_new,
    'USGS': common_daily_usgs,
    'GloFAS': common_daily_glofas
}, index=common_daily_nwm_new.index)

# Find the first occurrence of an NA
if combined_data.isna().any().any():
    first_na_index = combined_data[combined_data.isna().any(axis=1)].index.min()
    # Slice the DataFrame to exclude all rows after the first NA
    combined_data_cleaned = combined_data.loc[:first_na_index - pd.Timedelta(days=1)]
    print(f"Data truncated after the first occurrence of NA at index {first_na_index}.")
else:
    combined_data_cleaned = combined_data
    print("No NAs found in the DataFrame. No rows removed.")

# Optionally, save the cleaned data back to CSV
output_path_cleaned = "/data/muscat_data/jaguir26/project1_ucsc_phd/combined_streamflow_data_cleaned.csv"
combined_data_cleaned.to_csv(output_path_cleaned)
print(f"Cleaned data saved to {output_path_cleaned}")

# Display the first few rows of the cleaned DataFrame
print(combined_data_cleaned.head())
print(combined_data_cleaned.tail())


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Define the cutoff date
cutoff_date = pd.Timestamp('2022-12-20')

# Filter NWS3.0 and GloFAS data to stop at the cutoff date
filtered_nws3 = combined_data_cleaned['NWS3.0'].loc[:cutoff_date]
filtered_glofas = combined_data_cleaned['GloFAS'].loc[:cutoff_date]

# Plotting the data with USGS data across its full range
plt.figure(figsize=(14, 7))
plt.plot(filtered_nws3.index, filtered_nws3, label='NWS3.0', color='darkred')
plt.plot(combined_data_cleaned.index, combined_data_cleaned['USGS'], label='USGS', linestyle = 'dashed',  color='green', markersize = 0.5 )
plt.xlabel('Date')
plt.plot(filtered_glofas.index, filtered_glofas, label='GloFAS', color='orange')

# Add a vertical dashed line at the cutoff date
plt.axvline(cutoff_date, color='darkred', linestyle='--', linewidth=0.5, label=f'Cutoff Date: {cutoff_date}')

plt.title('Retros vs USGS')
plt.xlabel('Date')
plt.ylabel('NOT st - log(Discharge + 1)')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import pandas as pd

# Define the cutoff date
cutoff_date = pd.Timestamp('2022-12-01')

# Filter the DataFrame to only include data up to and including the cutoff date
final_data = combined_data_cleaned.loc[:cutoff_date]


### Show nws forecasts!
### Use forecasts for nws or glofas?

## NWS Forecast EDA

In [ ]:
CFSToCMS_CONVERSION_FACTOR = 0.0283168466

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.dates as mdates
import pickle
import pandas as pd
from datetime import datetime

def extract_forecast_data(filepath):
    """ Load data from a pickle file and organize it into a structured pandas DataFrame. """
    with open(filepath, 'rb') as file:
        data = pickle.load(file)

    # Prepare a list to collect all entries
    forecast_entries = []

    # Iterate through each key in the dictionary
    for key, value in data.items():
        parts = key.split('/')
        date_part = parts[0].split('.')[1]
        forecast_date = datetime.strptime(date_part, '%Y%m%d')
        ensemble_part = parts[1]

        if 'medium_range' in ensemble_part:
            if 'mem' in ensemble_part:
                ensemble_number = int(ensemble_part.split('mem')[1][0])
            else:
                ensemble_number = 1  # Assign to the first ensemble member if not specified
        else:
            ensemble_number = 1  # Default to 1 if no ensemble information is present

        lead_time = int(parts[2].split('f')[1].split('.')[0])

        # Append to the list as a tuple
        forecast_entries.append((forecast_date, ensemble_number, lead_time, value))

    # Create a DataFrame from the collected entries
    df = pd.DataFrame(forecast_entries, columns=['Date', 'Ensemble_Number', 'Lead_Time', 'Value'])

    return df

# Path to the 'results.pkl' file
pkl_file_path = '/data/muscat_data/jaguir26/project1_ucsc_phd/results.pkl'

# Extract data and print the DataFrame
forecast_df = extract_forecast_data(pkl_file_path)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.dates as mdates
import pandas as pd

def standardize_values(values):
    """ Standardize the given Pandas Series. """
    mean = values.mean()
    std = values.std()
    return (values - mean) / std

def plot_all_ensemble_forecasts(df, include_ensemble=True, ensemble_number=None, include_usgs=True, include_nws3=True, include_nws2=True):
    # Define cutoff dates and their descriptions
    cutoff_dates = {
        pd.Timestamp("2018-09-17"): "NWS1.0",
        pd.Timestamp("2019-06-19"): "NWS2.0",
        pd.Timestamp("2021-04-20"): "Hourly data",
        pd.Timestamp("2019-11-25"): "NWS2.1",
        pd.Timestamp("2023-01-10"): "SC22' flood",
        pd.Timestamp("2023-09-20"): "NWS3.0"
    }
    

    # Set up the figure
    plt.figure(figsize=(20, 7))

    # Create a color map and define colors based on ensemble numbers
    colormap = plt.cm.cividis

    ensemble_numbers = df['Ensemble_Number'].unique() if include_ensemble else []
    colors = {num: colormap(i / len(ensemble_numbers)) for i, num in enumerate(sorted(ensemble_numbers))}

    # Standardize forecast values
    # df['Standardized Transformed Value'] = standardize_values(np.log1p(df['Value']))
    df['Standardized Transformed Value'] = (np.log1p(df['Value']))

    # Plot each ensemble if requested
    if include_ensemble:
        ensembles_to_plot = [ensemble_number] if ensemble_number else ensemble_numbers
        for ensem_num in ensembles_to_plot:
            if ensem_num in colors:
                ensem_df = df[df['Ensemble_Number'] == ensem_num].sort_values(by=['Date', 'Lead_Time'])
                plt.plot(ensem_df['Date'], ensem_df['Standardized Transformed Value'], 
                        label=f'Ensemble {ensem_num}', color=colors[ensem_num], marker='.', linestyle='-', markersize=4, linewidth=0.5)

    # Plot additional series if indicated
    if include_nws3:
        plt.plot(filtered_nws3.index, filtered_nws3, label='NWS3.0', color='darkred')
    if include_nws2:
        # nws2_values = standardize_values(filtered_nwm_old['log_streamflow'])
        nws2_values = (filtered_nwm_old['log_streamflow'])
        plt.plot(filtered_nwm_old.index, nws2_values, label='NWM2.1', color='pink')
    if include_usgs:
        start_date = '2018-01-01'
        usgs = df_usgs.loc[start_date:].copy()
        # usgs_values = standardize_values(usgs['log_discharge_cms'])
        usgs_values = (usgs['log_discharge_cms'])
        plt.plot(usgs.index, usgs_values, label='USGS', linestyle='dashed', color='green', markersize=1)

    # Set plot titles and labels
    plt.title(' ')
    plt.xlabel('Date')
    plt.ylabel('st - log(discharge + 1)')
    plt.gca().xaxis.set_major_locator(mdates.YearLocator())
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.gca().xaxis.set_minor_locator(mdates.MonthLocator())
    plt.xticks(rotation=45)
    plt.grid(True, which='both', linestyle='--', linewidth=0.5)

    # Add cutoff lines with text annotations
    for date, description in cutoff_dates.items():
        plt.axvline(date, color='black', linestyle='--', linewidth=1)
        plt.text(date, plt.gca().get_ylim()[1], description, horizontalalignment='center', verticalalignment='bottom', rotation=0, color='black')

    plt.legend(title='Data Series', loc='upper left')
    plt.tight_layout()
    plt.show()


In [ ]:
plot_all_ensemble_forecasts(
    forecast_df,
    include_ensemble=False,
    ensemble_number=None,
    include_usgs=True,
    include_nws3=True,
    include_nws2=True
)
plot_all_ensemble_forecasts(
    forecast_df,
    include_ensemble=True,
    ensemble_number=1,
    include_usgs=True,
    include_nws3=False,
    include_nws2=False
)
plot_all_ensemble_forecasts(
    forecast_df,
    include_ensemble=True,
    ensemble_number=1,
    include_usgs=True,
    include_nws3=False,
    include_nws2=True
)
plot_all_ensemble_forecasts(
    forecast_df,
    include_ensemble=True,
    ensemble_number=1,
    include_usgs=True,
    include_nws3=True,
    include_nws2=True
)
plot_all_ensemble_forecasts(
    forecast_df,
    include_ensemble=True,
    ensemble_number=2,
    include_usgs=True,
    include_nws3=False,
    include_nws2=False
)
plot_all_ensemble_forecasts(
    forecast_df,
    include_ensemble=True,
    ensemble_number=3,
    include_usgs=True,
    include_nws3=False,
    include_nws2=False
)
plot_all_ensemble_forecasts(
    forecast_df,
    include_ensemble=True,
    ensemble_number=4,
    include_usgs=True,
    include_nws3=False,
    include_nws2=False
)
plot_all_ensemble_forecasts(
    forecast_df,
    include_ensemble=True,
    ensemble_number=5,
    include_usgs=True,
    include_nws3=False,
    include_nws2=False
)
plot_all_ensemble_forecasts(
    forecast_df,
    include_ensemble=True,
    ensemble_number=6,
    include_usgs=True,
    include_nws3=False,
    include_nws2=False
)
plot_all_ensemble_forecasts(
    forecast_df,
    include_ensemble=True,
    ensemble_number=7,
    include_usgs=True,
    include_nws3=False,
    include_nws2=False
)
plot_all_ensemble_forecasts(
    forecast_df,
    include_ensemble=True,
    ensemble_number=1,
    include_usgs=True,
    include_nws3=True,
    include_nws2=False
)
plot_all_ensemble_forecasts(
    forecast_df,
    include_ensemble=True,
    ensemble_number=5,
    include_usgs=True,
    include_nws3=True,
    include_nws2=False
)




## Creating training dataset

In [ ]:
end_date = '2022-12-25'

filtered_nwm_new = df_nwm_new.loc[:end_date].copy()
filtered_usgs = df_usgs.loc[:end_date].copy()
filtered_glofas = df_glofas.loc[:end_date].copy()


# Transform the data by adding 1 and then taking the logarithm
filtered_nwm_new['log_streamflow'] = np.log1p(filtered_nwm_new['streamflow'])
filtered_usgs['log_discharge_cms'] = np.log1p(filtered_usgs['discharge_cms'])
filtered_glofas['log_streamflow'] = np.log1p(filtered_glofas['Streamflow'])

# # Standardize the data
# filtered_nwm_new['std_streamflow'] = (filtered_nwm_new['log_streamflow'] - filtered_nwm_new['log_streamflow'].mean()) / filtered_nwm_new['log_streamflow'].std()
# filtered_usgs['std_discharge_cms'] = (filtered_usgs['log_discharge_cms'] - filtered_usgs['log_discharge_cms'].mean()) / filtered_usgs['log_discharge_cms'].std()
# filtered_glofas['std_streamflow'] = (filtered_glofas['log_streamflow'] - filtered_glofas['log_streamflow'].mean()) / filtered_glofas['log_streamflow'].std()

# # Resample data to daily frequency by averaging hourly values
# daily_nwm_new = filtered_nwm_new['std_streamflow'].resample('D').mean()
# daily_usgs = filtered_usgs['std_discharge_cms'].resample('D').mean()
# daily_glofas = filtered_glofas['std_streamflow'].resample('D').mean()

# Resample data to daily frequency by averaging hourly values
daily_nwm_new = filtered_nwm_new['log_streamflow'].resample('D').mean()
daily_usgs = filtered_usgs['log_discharge_cms'].resample('D').mean()
daily_glofas = filtered_glofas['log_streamflow'].resample('D').mean()

# Convert all datetime indices to timezone-naive
daily_nwm_new.index = daily_nwm_new.index.tz_localize(None)
daily_usgs.index = daily_usgs.index.tz_localize(None)
daily_glofas.index = daily_glofas.index.tz_localize(None)

# Find the maximum start date and the minimum end date among the datasets
start_date = max(daily_nwm_new.index.min(), daily_usgs.index.min(), daily_glofas.index.min())
end_date = min(daily_nwm_new.index.max(), daily_usgs.index.max(), daily_glofas.index.max())

# Align all datasets to the common date range
common_daily_nwm_new = daily_nwm_new.loc[start_date:end_date]
common_daily_usgs = daily_usgs.loc[start_date:end_date]
common_daily_glofas = daily_glofas.loc[start_date:end_date]

# Create a new DataFrame combining all time series
combined_data = pd.DataFrame({
    'NWS3.0': common_daily_nwm_new,
    'USGS': common_daily_usgs,
    'GloFAS': common_daily_glofas
}, index=common_daily_nwm_new.index)

# Find the first occurrence of an NA
if combined_data.isna().any().any():
    first_na_index = combined_data[combined_data.isna().any(axis=1)].index.min()
    # Slice the DataFrame to exclude all rows after the first NA
    combined_data_cleaned = combined_data.loc[:first_na_index - pd.Timedelta(days=1)]
    print(f"Data truncated after the first occurrence of NA at index {first_na_index}.")
else:
    combined_data_cleaned = combined_data
    print("No NAs found in the DataFrame. No rows removed.")

# Optionally, save the cleaned data back to CSV
output_path_cleaned = "/data/muscat_data/jaguir26/project1_ucsc_phd/combined_streamflow_data_cleaned.csv"
combined_data_cleaned.to_csv(output_path_cleaned)
print(f"Cleaned data saved to {output_path_cleaned}")

# Display the first few rows of the cleaned DataFrame
print(combined_data_cleaned.head())
print(combined_data_cleaned.tail())

# Specify the path for the CSV file
output_path = "/data/muscat_data/jaguir26/project1_ucsc_phd/combined_streamflow_data.csv"
# Save the DataFrame to a CSV file
combined_data.to_csv(output_path)
print(f"Data saved to {output_path}")

In [ ]:
end_date = '2023-12-25'

filtered_nwm_new = df_nwm_new.loc[:end_date].copy()
filtered_usgs = df_usgs.loc[:end_date].copy()
filtered_glofas = df_glofas.loc[:end_date].copy()

# Transform the data by adding 1 and then taking the logarithm
filtered_nwm_new['log_streamflow'] = np.log1p(filtered_nwm_new['streamflow'])
filtered_usgs['log_discharge_cms'] = np.log1p(filtered_usgs['discharge_cms'])
filtered_glofas['log_streamflow'] = np.log1p(filtered_glofas['Streamflow'])

# Standardize the data
filtered_nwm_new['std_streamflow'] = (filtered_nwm_new['log_streamflow'] - filtered_nwm_new['log_streamflow'].mean()) / filtered_nwm_new['log_streamflow'].std()
filtered_usgs['std_discharge_cms'] = (filtered_usgs['log_discharge_cms'] - filtered_usgs['log_discharge_cms'].mean()) / filtered_usgs['log_discharge_cms'].std()
filtered_glofas['std_streamflow'] = (filtered_glofas['log_streamflow'] - filtered_glofas['log_streamflow'].mean()) / filtered_glofas['log_streamflow'].std()

# # Resample data to daily frequency by averaging hourly values
# daily_nwm_new = filtered_nwm_new['std_streamflow'].resample('D').mean()
# daily_usgs = filtered_usgs['std_discharge_cms'].resample('D').mean()
# daily_glofas = filtered_glofas['std_streamflow'].resample('D').mean()

# Resample data to daily frequency by averaging hourly values
daily_nwm_new = filtered_nwm_new['log_streamflow'].resample('D').mean()
daily_usgs = filtered_usgs['log_discharge_cms'].resample('D').mean()
daily_glofas = filtered_glofas['log_streamflow'].resample('D').mean()

# Convert all datetime indices to timezone-naive
daily_nwm_new.index = daily_nwm_new.index.tz_localize(None)
daily_usgs.index = daily_usgs.index.tz_localize(None)
daily_glofas.index = daily_glofas.index.tz_localize(None)

# Find the maximum start date and the minimum end date among the datasets
start_date = max(daily_nwm_new.index.min(), daily_usgs.index.min(), daily_glofas.index.min())

common_daily_usgs = daily_usgs.loc[start_date:]

# Optionally, save the cleaned data back to CSV
output_path_cleaned = "/data/muscat_data/jaguir26/project1_ucsc_phd/combined_streamflow_data_cleaned_extra.csv"
common_daily_usgs.to_csv(output_path_cleaned)
print(f"Cleaned data saved to {output_path_cleaned}")



## Weighted averages across lead times

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datetime import timedelta
import matplotlib.dates as mdates
import pickle
from datetime import datetime

def extract_forecast_data(filepath):
    with open(filepath, 'rb') as file:
        data = pickle.load(file)
    forecast_entries = []
    for key, value in data.items():
        parts = key.split('/')
        date_part = parts[0].split('.')[1]
        forecast_date = datetime.strptime(date_part, '%Y%m%d')
        ensemble_part = parts[1]
        ensemble_number = int(ensemble_part.split('mem')[1]) if 'mem' in ensemble_part else 1
        lead_time = int(parts[2].split('f')[1].split('.')[0])
        forecast_entries.append((forecast_date, ensemble_number, lead_time, value))
    return pd.DataFrame(forecast_entries, columns=['Date', 'Ensemble_Number', 'Lead_Time', 'Value'])


In [ ]:

# Load data
pkl_file_path = '/data/muscat_data/jaguir26/project1_ucsc_phd/results.pkl'
forecast_df = extract_forecast_data(pkl_file_path)

# Add Target_Time column and calculate weights
forecast_df['Target_Time'] = forecast_df['Date'] + pd.to_timedelta(forecast_df['Lead_Time'], unit='h')
forecast_df['Weight'] = 1 / forecast_df['Lead_Time']
forecast_df['Standardized Transformed Value'] = (np.log1p(forecast_df['Value']) - np.log1p(forecast_df['Value']).mean()) / np.log1p(forecast_df['Value']).std()
forecast_df['Weighted Standardized Value'] = forecast_df['Standardized Transformed Value'] * forecast_df['Weight']

# Sum weights and weighted values
grouped = forecast_df.groupby(['Target_Time', 'Ensemble_Number'])
sum_weights = grouped['Weight'].sum()
sum_weighted_values = grouped['Weighted Standardized Value'].sum()

# Calculate weighted average without apply()
weighted_averages = (sum_weighted_values / sum_weights).reset_index()
weighted_averages.columns = ['Target_Time', 'Ensemble_Number', 'Weighted Average']

# Pivot for daily averages
daily_averages = weighted_averages.pivot(index='Target_Time', columns='Ensemble_Number', values='Weighted Average').resample('D').mean()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np

def standardize_values(values):
    """ Standardize the given Pandas Series. """
    mean = values.mean()
    std = values.std()
    return (values - mean) / std

def plot_ensemble_forecasts(daily_averages, df_usgs, ensemble_list=None, start_date='2018-01-01', end_date=None):
    plt.figure(figsize=(20, 8))
    colormap = plt.cm.cividis
    
    # Define the maximum expected number of ensemble members and generate colors
    max_ensemble = 10
    colors = {num: colormap(i / max_ensemble) for i, num in enumerate(range(1, max_ensemble + 1))}

    # Parse dates if provided as strings
    start_date = pd.to_datetime(start_date)
    if end_date:
        end_date = pd.to_datetime(end_date)

    # Filter data by date range
    if end_date:
        daily_averages = daily_averages.loc[pd.to_datetime('2022-12-26'):pd.to_datetime('2023-01-06')]
        df_usgs = df_usgs.loc[start_date:end_date]
    else:
        daily_averages = daily_averages.loc[pd.to_datetime('2022-12-26'):pd.to_datetime('2023-01-06')]
        df_usgs = df_usgs.loc[start_date:]

    # usgs_values = standardize_values(df_usgs['log_discharge_cms'])
    usgs_values = (df_usgs['log_discharge_cms'])
    plt.plot(df_usgs.index, usgs_values, label='USGS', linestyle='dashed', color='green', markersize=1)

    ft = combined_data_cleaned['NWS3.0'].loc[start_date:pd.to_datetime('2022-12-25')].copy()
    # a = standardize_values(np.log1p(ft))
    # a = standardize_values((ft))
    a = ((ft))
    plt.plot(ft.index, a, label='NWS3.0', linestyle='solid', color='darkred', markersize=1)
    
    ft = combined_data_cleaned['GloFAS'].loc[start_date:pd.to_datetime('2022-12-25')].copy()
    # a = standardize_values(np.log1p(ft))
    a = ((ft))
    a = ((ft))
    plt.plot(ft.index, a, label='GloFAS', linestyle='solid', color='darkorange', markersize=1)
              

    # Plot each selected ensemble
    if ensemble_list is None:
        ensemble_list = daily_averages.columns
    for ensemble in sorted(ensemble_list):
        if ensemble in daily_averages.columns:
            plt.plot(daily_averages.index, daily_averages[ensemble], label=f'Ensemble {ensemble}', color=colors[int(ensemble)], marker='.', linestyle='-')

    # Define cutoff dates and add annotations if within the date range
    cutoff_dates = {
        pd.Timestamp("2018-09-17"): "NWS1.0",
        pd.Timestamp("2019-06-19"): "NWS2.0",
        pd.Timestamp("2021-04-20"): "Hourly data",
        pd.Timestamp("2019-11-25"): "NWS2.1",
        pd.Timestamp("2023-01-10"): "SC22' flood",
        pd.Timestamp("2023-09-20"): "NWS3.0",
        pd.Timestamp("2022-12-25"): "Training Cutoff"
    }
    for date, description in cutoff_dates.items():
        if start_date <= date <= (end_date if end_date else date):
            plt.axvline(date, color='black', linestyle='--', linewidth=1)
            plt.text(date, plt.gca().get_ylim()[1], description, horizontalalignment='center', verticalalignment='bottom', rotation=0, color='black')

    plt.title('')
    plt.xlabel('Date')
    plt.ylabel('Daily Weighted Average of Log Values')
    plt.legend(title='Ensemble Number')
    plt.gca().xaxis.set_major_locator(mdates.YearLocator())
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.gca().xaxis.set_minor_locator(mdates.MonthLocator())
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

plot_ensemble_forecasts(daily_averages, df_usgs, ensemble_list=[1,2,3,4,5,6,7], start_date='2022-11-20', end_date='2023-02-01')

Bring glofas forecats and retro! 

Check that retros and forecats are in the right date, like not displaced. 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def calculate_cross_correlation(data, column1, column2, max_lag):
    """Calculate cross-correlation between two columns of a DataFrame up to a maximum lag."""
    correlations = {}
    for lag in range(-max_lag, max_lag + 1):
        shifted = data[column1].shift(lag)
        correlation = shifted.corr(data[column2])
        correlations[lag] = correlation
    return correlations

# Load your data
combined_data_cleaned = pd.read_csv('/data/muscat_data/jaguir26/project1_ucsc_phd/combined_streamflow_data_cleaned.csv', index_col=0, parse_dates=True)

# Specify the maximum number of lags (days) you want to consider
max_lag = 30

# Define pairs to analyze
pairs = [
    ('NWS3.0', 'USGS'),
    ('NWS3.0', 'GloFAS'),
    ('USGS', 'GloFAS')
]

# Figure setup for plotting
fig, axes = plt.subplots(nrows=len(pairs), ncols=1, figsize=(10, 15))
fig.tight_layout(pad=6.0)

# Calculate and plot cross-correlation for each pair
for i, (series1, series2) in enumerate(pairs):
    cross_correlation = calculate_cross_correlation(combined_data_cleaned, series1, series2, max_lag)
    max_corr_lag = max(cross_correlation, key=cross_correlation.get)
    max_corr_value = cross_correlation[max_corr_lag]

    print(f"The highest cross-correlation between {series1} and {series2} is at lag {max_corr_lag} days with a correlation of {max_corr_value:.3f}.")

    lags = list(cross_correlation.keys())
    correlation_values = list(cross_correlation.values())
    
    axes[i].stem(lags, correlation_values)
    axes[i].set_title(f'Cross-Correlation between {series1} and {series2}')
    axes[i].set_xlabel('Lag (days)')
    axes[i].set_ylabel('Correlation coefficient')
    axes[i].grid(True)

plt.show()


In [ ]:
os.chdir('/data/muscat_data/jaguir26/projects/')
# Define a function to create a directory for a given site code
def create_directory_ID(base_path, folder_id):
    directory_path = os.path.join(base_path, f"{folder_id}")
    if not os.path.exists(directory_path):
        os.makedirs(directory_path)
    return directory_path

# Define a function to create a directory for a given site code
def create_directory_NAME(base_path, folder_name):
    directory_path = os.path.join(base_path, folder_name)
    if not os.path.exists(directory_path):
        os.makedirs(directory_path)
    return directory_path
base_path = os.getcwd()
directory_path_project = create_directory_NAME(base_path, 'Project')

base_path = directory_path_project
directory_path_input = create_directory_NAME(base_path, 'Input')
directory_path_output = create_directory_NAME(base_path, 'Output')

base_path = directory_path_input
directory_path_input_id_river = create_directory_NAME(base_path, 'ID_River')
directory_path_nws_coord = create_directory_NAME(base_path, 'NWS-Coordinates')
directory_path_retro = create_directory_NAME(base_path, 'Retrospective_Analysis')
directory_path_input_exAL = create_directory_NAME(base_path, 'exAL')

base_path = directory_path_retro
directory_path_retro_nws = create_directory_NAME(base_path, 'NWS')
directory_path_retro_glofas = create_directory_NAME(base_path, 'GLOFAS')

base_path = directory_path_input_exAL
directory_path_input_covariates = create_directory_NAME(base_path, 'covariates')
directory_path_input_model = create_directory_NAME(base_path, 'model_outputs')
directory_path_input_parameters = create_directory_NAME(base_path, 'parameters')
directory_path_input_R_script= create_directory_NAME(base_path, 'R_script')

base_path = directory_path_output
directory_path_output_id_river = create_directory_NAME(base_path, 'ID_River')

# Constants
CFSToCMS_CONVERSION_FACTOR = 0.0283168466

# Define time range
start_usgs = '1979-01-01'

today = datetime.today()
# Format it as YYYY-MM-DD
end_usgs = today.strftime('%Y-%m-%d')

# Convert end date to datetime object and get start date for forecast
end_usgs_datetime = datetime.strptime(end_usgs, '%Y-%m-%d')
start_forecast_datetime = end_usgs_datetime - timedelta(days=1)
start_forecast = start_forecast_datetime.strftime('%Y-%m-%d')

# Specify the USGS site code
site_code = '11160500'

# Fetch daily data for the site
df = nwis.get_record(sites=site_code, service='dv', parameterCd='00060', statCd='00003',
                     start=start_usgs, end=end_usgs)

# Log-transform the flow data; we add 1 to handle cases where the value is 0
df['log_discharge'] = np.log(df['00060_Mean'].astype(float) + 1)
# Keep only the relevant column
df = df[['log_discharge']]
# Reverse the log transformation to get back the discharge in cfs
df['discharge_cfs'] = np.exp(df['log_discharge']) - 1
# Convert discharge from cfs to cms
df['discharge_cms'] = df['discharge_cfs'] * CFSToCMS_CONVERSION_FACTOR
# Optionally, log-transform the discharge in cms
df['log_discharge_cms'] = np.log(df['discharge_cms'] + 1)

# Fetch metadata for the USGS site without specifying fields
site_info = nwis.get_record(sites=site_code, service='site')
station_name = site_info['station_nm'][0] 

# Extract latitude and longitude
latitude = float(site_info['dec_lat_va'][0])
longitude = float(site_info['dec_long_va'][0])

# Combine into a single target_location tuple
target_location = (latitude, longitude)
target_lat, target_lon = target_location
print(f"The coordinates for site {site_code} are {target_location}")

start_forecast = '2022-12-26'

def filter_dataframe(df, start_forecast, pre_days, post_days):
    """
    Filter the DataFrame based on the forecast start date and the number of days before and after.

    Parameters:
    - df: DataFrame to filter
    - start_forecast: Start date of the forecast (string in 'YYYY-MM-DD' format)
    - pre_days: Number of days before the forecast start date
    - post_days: Number of days after the forecast start date

    Returns:
    - Filtered DataFrame
    """
    # Convert start forecast to datetime object
    start_forecast_dt = datetime.strptime(start_forecast, '%Y-%m-%d')

    # Calculate the start and end dates for filtering
    start_filter_dt = start_forecast_dt - timedelta(days=pre_days)
    end_filter_dt = start_forecast_dt + timedelta(days=post_days)

    # Convert dates back to string
    start_filter_str = start_filter_dt.strftime('%Y-%m-%d')
    end_filter_str = end_filter_dt.strftime('%Y-%m-%d')

    # Filter the DataFrame
    return df.loc[start_filter_str:end_filter_str]

# Applying the function to different forecast types
df_filtered = filter_dataframe(df, start_forecast, 10, 45)
df_filtered_s = filter_dataframe(df, start_forecast, 30, 270)
df_filtered_rf = filter_dataframe(df, start_forecast, 30, 270)

def filter_dataframe(df, start_forecast, pre_days, post_days):
    start_forecast_dt = datetime.strptime(start_forecast, '%Y-%m-%d')
    start_filter_dt = start_forecast_dt - timedelta(days=pre_days)
    end_filter_dt = start_forecast_dt + timedelta(days=post_days)
    return df.loc[start_filter_dt.strftime('%Y-%m-%d'):end_filter_dt.strftime('%Y-%m-%d')]

def convert_date_format(date_str):
    """
    Convert a date string from 'YYYY-MM-DD' to 'YYYYMMDD'
    
    Parameters:
        date_str (str): Date string in 'YYYY-MM-DD' format
    
    Returns:
        str: Date string in 'YYYYMMDD' format
    """
    return date_str.replace("-", "")

# Example usage
start_forecast_joint = convert_date_format(start_forecast)

# Input Dependent
base_path = directory_path_input_id_river
directory_path_input_ID = create_directory_ID(base_path, site_code)
base_path = directory_path_input_ID
directory_path_input_id_river_frsct_date = create_directory_NAME(base_path, 'Start_Forecast_Date')
directory_path_input_id_river_csv = create_directory_NAME(base_path, 'USGS_csv')
base_path = directory_path_input_id_river_frsct_date
directory_input_id_frsc = create_directory_ID(base_path, start_forecast_joint)

# Output Dependent
base_path = directory_path_output_id_river
directory_path_output_ID = create_directory_ID(base_path, site_code)
base_path = directory_path_output_ID
directory_path_output_id_river_frsct_date = create_directory_NAME(base_path, 'Start_Forecast_Date')
base_path = directory_path_output_id_river_frsct_date
directory_output_id_frsc  = create_directory_ID(base_path, start_forecast_joint)

# Forecasts (Inputs)
base_path = directory_input_id_frsc
directory_glofas_frsc = create_directory_NAME(base_path, 'GLOFAS_Forecasts')
directory_nws_frsc = create_directory_NAME(base_path, 'NWS_Forecasts')

base_path = directory_glofas_frsc
directory_glofas_frsc_med = create_directory_NAME(base_path, 'Medium_Range')
directory_glofas_frsc_seas = create_directory_NAME(base_path, 'Seasonal_Range')
directory_glofas_frsc_rf = create_directory_NAME(base_path, 'Reforecast_Range')

base_path = directory_nws_frsc
directory_nws_frsc_med = create_directory_NAME(base_path, 'Medium_Range')
directory_nws_frsc_long = create_directory_NAME(base_path, 'Long_Range')

# Model outputs and figures (Outputs)
base_path = directory_output_id_frsc
directory_path_model_output = create_directory_NAME(base_path, 'exAL_output')
directory_path_figures= create_directory_NAME(base_path, 'Figures')

def choose_hydrological_model(start_forecast):
    lisflood_start_date = datetime.strptime('2021-05-26', '%Y-%m-%d')
    htessel_lisflood_end_date = datetime.strptime('2022-09-13', '%Y-%m-%d')
    forecast_date = datetime.strptime(start_forecast, '%Y-%m-%d')
    if forecast_date < lisflood_start_date:
        return 'htessel_lisflood'
    else:
        return 'lisflood'


In [ ]:

def retrieve_glofas_data_medium(
    start_forecast,
    target_location,
    buffer=1,
    system_version='operational',
    product_type=['control_forecast','ensemble_perturbed_forecasts'],
    variable='river_discharge_in_the_last_24_hours',
    leadtime_hour= [str(i) for i in range(24, 721, 24)],
    target_folder=directory_glofas_frsc_med
):
        # Data availability check
    if datetime.strptime(start_forecast, '%Y-%m-%d') < datetime(2019, 11, 11):
        print("No data available before Nov 11, 2019 for medium-range forecasts.")
        return None
    
    hydrological_model = choose_hydrological_model(start_forecast)
    dt = datetime.strptime(start_forecast, '%Y-%m-%d')
    year, month, day = dt.year, dt.month, dt.day
    latitude, longitude = target_location
    area = [latitude + buffer, longitude - buffer, latitude - buffer, longitude + buffer]
    short_product_type = '_'.join(''.join(word[0] for word in item.split('_')) for item in product_type)
    short_variable = ''.join(word[0] for word in variable.split('_'))
    file_name_elements = [
        system_version[:3], 
        hydrological_model[:3], 
        short_product_type,
        short_variable,
        str(year), str(month).zfill(2), str(day).zfill(2),
        f"lt_{leadtime_hour[0]}_to_{leadtime_hour[-1]}", 
        f"area_{round(area[0], 2)}_{round(area[1], 2)}_{round(area[2], 2)}_{round(area[3], 2)}"
    ]
    file_name = '_'.join(file_name_elements) + '.grib'
    output_file = os.path.join(target_folder, file_name)
    if os.path.exists(output_file):
        print(f"File already exists: {output_file}")
        return output_file
    c = cdsapi.Client()
    retrieval_params = {
        'system_version': system_version,
        'hydrological_model': hydrological_model,
        'product_type': product_type,
        'variable': variable,
        'year': str(year),
        'month': str(month).zfill(2),
        'day': str(day).zfill(2),
        'leadtime_hour': leadtime_hour,
        'format': 'grib',
        'area': area
    }
    c.retrieve('cems-glofas-forecast', retrieval_params, output_file)
    print(f"Retrieval completed. Output file saved to: {output_file}")
    return output_file


In [ ]:

def retrieve_glofas_data_seasonal(
    start_forecast,
    target_location,
    buffer=1,
    system_version='operational',
    product_type=['control_forecast', 'ensemble_perturbed_forecasts'],
    variable='river_discharge_in_the_last_24_hours',
    leadtime_hour = [str(i) for i in range(24, 5161, 24)],
    target_folder=directory_glofas_frsc_seas,
    leadtime_chunk_size=50
):
    
        # Data availability check
    if datetime.strptime(start_forecast, '%Y-%m-%d') < datetime(2020, 12, 1):
        print("No data available before Dec 01, 2020 for seasonal forecasts.")
        return None


    hydrological_model = choose_hydrological_model(start_forecast)
    dt = datetime.strptime(start_forecast, '%Y-%m-%d')
    year, month, day = dt.year, dt.month, dt.day
    latitude, longitude = target_location
    area = [latitude + buffer, longitude - buffer, latitude - buffer, longitude + buffer]
    c = cdsapi.Client()
    chunks = [leadtime_hour[i:i + leadtime_chunk_size] for i in range(0, len(leadtime_hour), leadtime_chunk_size)]
    output_files = []
    for chunk in chunks:
        short_product_type = '_'.join(''.join(word[0] for word in item.split('_')) for item in product_type)
        short_variable = ''.join(word[0] for word in variable.split('_'))
        file_name_elements = [
            system_version[:3],
            hydrological_model[:3],
            short_product_type,
            short_variable,
            str(year), str(month).zfill(2), str(day).zfill(2),
            f"lt_{chunk[0]}_to_{chunk[-1]}",
            f"area_{round(area[0], 2)}_{round(area[1], 2)}_{round(area[2], 2)}_{round(area[3], 2)}"
        ]
        chunk_filename = 'seas_' + '_'.join(file_name_elements) + '.grib'
        output_file = os.path.join(target_folder, chunk_filename)
        if os.path.exists(output_file):
            print(f"File already exists: {output_file}")
            output_files.append(output_file)
            continue
        retrieval_params = {
            'system_version': system_version,
            'hydrological_model': hydrological_model,
            'product_type': product_type,
            'variable': variable,
            'year': str(year),
            'month': str(month).zfill(2),
            'day': str(day).zfill(2),
            'leadtime_hour': chunk,
            'format': 'grib',
            'area': area
        }
        c.retrieve('cems-glofas-seasonal', retrieval_params, output_file)
        print(f"Retrieval completed. Output file saved to: {output_file}")
        output_files.append(output_file)
    return output_files


In [ ]:

def retrieve_glofas_reforecasts(
    start_forecast,
    target_location,
    buffer=1,
    system_version='version_4_0',
    variable='river_discharge_in_the_last_24_hours',
    target_folder=directory_glofas_frsc_rf,
    leadtime_chunk_size=50
):
    # Extract year and month from start_forecast
    dt = datetime.strptime(start_forecast, '%Y-%m-%d')
    hyear, hmonth = dt.year, dt.month

    # Create a bounding box around the target location
    latitude, longitude = target_location
    area = [latitude + buffer, longitude - buffer, latitude - buffer, longitude + buffer]

    # Initialize the CDS API client
    c = cdsapi.Client()

    # Define all leadtime_hours 
    leadtime_hour = [str(i) for i in range(24, 5160, 24)]  

    # Divide the leadtime_hour into chunks
    chunks = [leadtime_hour[i:i + leadtime_chunk_size] for i in range(0, len(leadtime_hour), leadtime_chunk_size)]

    output_files = []  # List to store the paths of downloaded files

    for chunk in chunks:
        # Shorten the elements for the filename
        short_variable = ''.join(word[0] for word in variable.split('_'))

        # Create a shorter filename
        file_name_elements = [
            system_version[:3],
            'lis',  # Short for 'lisflood'
            short_variable,
            str(hyear), str(hmonth).zfill(2),
            f"lt_{chunk[0]}_to_{chunk[-1]}",
            f"area_{round(area[0], 2)}_{round(area[1], 2)}_{round(area[2], 2)}_{round(area[3], 2)}"
        ]
        chunk_filename = 'reforc_' + '_'.join(file_name_elements) + '.grib'

        # Construct the full path of the output file
        output_file = os.path.join(target_folder, chunk_filename)

        # Check if the file already exists
        if os.path.exists(output_file):
            print(f"File already exists: {output_file}")
            output_files.append(output_file)
            continue

        # Construct the retrieval parameters
        retrieval_params = {
            'system_version': system_version,
            'hydrological_model': 'lisflood',
            'variable': variable,
            'hyear': str(hyear),
            'hmonth': str(hmonth).zfill(2),
            'leadtime_hour': chunk,
            'format': 'grib',
            'area': area
        }

        # Perform the retrieval
        c.retrieve('cems-glofas-seasonal-reforecast', retrieval_params, output_file)
        print(f"Retrieval completed. Output file saved to: {output_file}")
        output_files.append(output_file)

    return output_files


# Call the function with a small leadtime_chunk_size for testing
# output_files_rf = retrieve_glofas_reforecasts(start_forecast, target_location, leadtime_chunk_size=120)

def get_common_prefix(strings):
    if not strings:
        return ""
    
    prefix = os.path.commonprefix(strings)
    # Remove the last part to get up to the last common '_'
    prefix = '_'.join(prefix.split('_')[:-1])
    return prefix

def merge_grib_files(output_files, dir_path):
    # Check if output_files is defined and contains file paths
    if not output_files or not isinstance(output_files, list) or not all(isinstance(file, str) for file in output_files):
        print("No valid output files provided for merging.")
        return None

    common_prefix = get_common_prefix(output_files)

    # Ensure there is a common prefix before proceeding
    if not common_prefix:
        print("No common prefix found for the files. Merging aborted.")
        return None

    # Define the name of the merged file based on the common prefix
    merged_file_name = f"{common_prefix}_merged.grib"
    merged_file_name = os.path.join(dir_path, merged_file_name)

    merge_command = ["grib_copy"] + output_files + [merged_file_name]
    subprocess.run(merge_command)

    print(f"Merged file saved to: {merged_file_name}")
    return merged_file_name


In [ ]:

import os
import subprocess

# Other imports and code ...

def merge_grib_files(output_files, dir_path):
    # Check if output_files is defined and contains file paths
    if not output_files or not isinstance(output_files, list) or not all(isinstance(file, str) for file in output_files):
        print("No valid output files provided for merging.")
        return None

    common_prefix = get_common_prefix(output_files)

    # Ensure there is a common prefix before proceeding
    if not common_prefix:
        print("No common prefix found for the files. Merging aborted.")
        return None

    # Define the name of the merged file based on the common prefix
    merged_file_name = f"{common_prefix}_merged.grib"
    merged_file_name = os.path.join(dir_path, merged_file_name)

    # Use the full path to grib_copy
    grib_copy_path = os.path.expanduser("~/local/bin/grib_copy")
    merge_command = [grib_copy_path] + output_files + [merged_file_name]
    
    result = subprocess.run(merge_command, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"Error occurred: {result.stderr}")
        return None

    print(f"Merged file saved to: {merged_file_name}")
    return merged_file_name

# # Call the function with a small leadtime_chunk_size for testing
# output_files_rf = retrieve_glofas_reforecasts(start_forecast, target_location, leadtime_chunk_size=120)
# merged_file_name_rf = merge_grib_files(output_files_rf, directory_glofas_frsc_rf)
# # Call the function with a small leadtime_chunk_size for testing
# output_files_s = retrieve_glofas_data_seasonal(start_forecast, target_location, leadtime_chunk_size=120)
# merged_file_name_s = merge_grib_files(output_files_s, directory_glofas_frsc_seas)
# Example of how to call the function
grib_file_path = retrieve_glofas_data_medium(start_forecast=start_forecast, target_location=target_location)


In [ ]:


def process_grib_file(file_path):
    if file_path is None or not os.path.exists(file_path):
        print(f"File does not exist or path is None: {file_path}")
        return None, None, None

    grbs = pygrib.open(file_path)
    unique_lead_times = set()
    unique_ensemble_members = set()

    for grb in grbs:
        unique_lead_times.add(grb['forecastTime'])
        unique_ensemble_members.add(grb['perturbationNumber'])

    grbs.seek(0)
    return grbs, unique_lead_times, unique_ensemble_members

# Check and process each GRIB file
grbs, unique_lead_times, unique_ensemble_members = process_grib_file(grib_file_path)
# grbs_s, unique_lead_times_s, unique_ensemble_members_s = process_grib_file(merged_file_name_s)
# grbs_rf, unique_lead_times_rf, unique_ensemble_members_rf = process_grib_file(merged_file_name_rf)


In [ ]:

# Define function to convert longitude
def convert_longitude_to_neg_180_180(lon):
    if lon > 180:
        return lon - 360
    return lon

def convert_longitude_to_0_360(lon):
    if lon < 0:
        return lon + 360
    return lon


# Function to find the closest coordinates
def find_closest_coordinates(grbs, target_location):
    if grbs is None:
        return None, None, None, None

    min_distance = float('inf')
    closest_coordinates = None
    closest_i = None  # Initialize variables to store the indices
    closest_j = None

    for grb in grbs:
        if grb['perturbationNumber'] == 0:
            latitudes, longitudes = grb.latlons()
            for i in range(len(latitudes)):
                for j in range(len(longitudes[0])):
                    coord = (latitudes[i][j], convert_longitude_to_neg_180_180(longitudes[i][j]))
                    distance = haversine(target_location, coord, unit=Unit.METERS)
                    if distance < min_distance:
                        min_distance = distance
                        closest_coordinates = coord
                        closest_i = i  # Store the index i
                        closest_j = j  # Store the index j
            break
    
    # Convert coordinates if needed
    closest_coordinates_glofas = None
    if closest_coordinates:
        closest_coordinates_glofas = (closest_coordinates[0], convert_longitude_to_0_360(closest_coordinates[1]))
    
    return closest_coordinates_glofas, closest_i, closest_j, min_distance

# Process for each GRIB file
closest_coordinates_glofas, closest_i, closest_j, min_distance = find_closest_coordinates(grbs, target_location)
if closest_coordinates_glofas:
    print("Closest Coordinates in GloFAS format:", closest_coordinates_glofas)
    # print("Indices in the GloFAS grid:", closest_i, closest_j)

# closest_coordinates_glofas_s, closest_i_s, closest_j_s, min_distance_s = find_closest_coordinates(grbs_s, target_location)
# if closest_coordinates_glofas_s:
#     print("Closest Coordinates in GloFAS format (Seasonal):", closest_coordinates_glofas_s)
#     # print("Indices in the GloFAS grid:", closest_i_s, closest_j_s)

# closest_coordinates_glofas_rf, closest_i_rf, closest_j_rf, min_distance_rf = find_closest_coordinates(grbs_rf, target_location)
# if closest_coordinates_glofas_rf:
#     print("Closest Coordinates in GloFAS format (Reforecast):", closest_coordinates_glofas_rf)
#     # print("Indices in the GloFAS grid:", closest_i_rf, closest_j_rf)


In [ ]:

# Function to process GloFAS data from GRIB files
def process_glofas_data(grbs, closest_indices):
    if grbs is None or closest_indices is None:
        print("GRIB file or closest indices not provided.")
        return {}, np.array([])

    closest_i, closest_j = closest_indices
    glofas_data = {}

    for grb in grbs:
        lead_time = grb['forecastTime']
        ensemble_member = grb['perturbationNumber']
        data_value = grb.values[closest_i, closest_j]
        glofas_data[(ensemble_member, lead_time)] = data_value

    grbs.close()

    # Sorting and indexing
    sorted_ensemble_members = sorted(set(val[0] for val in glofas_data.keys()))
    sorted_lead_times = sorted(set(val[1] for val in glofas_data.keys()))
    ensemble_member_to_idx = {em: idx for idx, em in enumerate(sorted_ensemble_members)}
    lead_time_to_idx = {lt: idx for idx, lt in enumerate(sorted_lead_times)}

    # Initializing and populating the array
    glofas_array = np.zeros((len(sorted_ensemble_members), len(sorted_lead_times)))
    for (ensemble_member, lead_time), value in glofas_data.items():
        i = ensemble_member_to_idx[ensemble_member]
        j = lead_time_to_idx[lead_time]
        glofas_array[i, j] = value

    return glofas_data, glofas_array


In [ ]:

# Process each GRIB file
glofas_data, glofas_array = process_glofas_data(grbs, (closest_i, closest_j))
# glofas_data_s, glofas_array_s = process_glofas_data(grbs_s, (closest_i_s, closest_j_s))
# glofas_data_rf, glofas_array_rf = process_glofas_data(grbs_rf, (closest_i_rf, closest_j_rf))

def plot_ensemble_data(array, start_forecast, dir_path, title_suffix, file_suffix):
    if array.size == 0:
        print(f"No data to plot for {title_suffix}.")
        return

    num_ensemble_members = array.shape[0]
    file_name = f"glofas_{file_suffix}_{start_forecast}_{num_ensemble_members}.png"
    file_path = os.path.join(dir_path, file_name)

    if os.path.exists(file_path):
        print(f"File {file_name} already exists.")
        return

    start_forecast_date = pd.Timestamp(start_forecast)
    lead_times_hours = np.arange(array.shape[1]) * 24  # Assuming 24-hour intervals
    lead_times_timedelta = pd.to_timedelta(lead_times_hours, unit='h')
    ensemble_timestamps = np.array(start_forecast_date + lead_times_timedelta)
    mean_ensemble = np.mean(array, axis=0)

    plt.figure(figsize=(20, 6))
    for i in range(num_ensemble_members):
        ensemble_member_data = np.array(array[i, :])
        plt.plot(ensemble_timestamps, np.log(ensemble_member_data + 1), color='orange', alpha=0.2)
    
    plt.plot(ensemble_timestamps, np.log(mean_ensemble + 1), color='darkorange', linewidth=2.5, label='Mean Ensemble')

    plt.xlabel('Time')
    plt.ylabel('Log Discharge (cms)')
    plt.title(f'Ensemble Members and Mean Ensemble - {title_suffix}')
    plt.grid(True)
    # plt.legend()

    plt.savefig(file_path)
    plt.close()


# Plotting for each array
plot_ensemble_data(glofas_array, start_forecast, directory_path_figures, "Medium Range", "medium")
# plot_ensemble_data(glofas_array_s, start_forecast, directory_path_figures, "Seasonal", "seasonal")
# plot_ensemble_data(glofas_array_rf, start_forecast, directory_path_figures, "Reforecast", "reforecast")

def plot_glofas_forecasts_with_usgs(glofas_arrays, usgs_data, start_forecast, dir_path, plot_titles, file_suffixes):
    for glofas_array, plot_title, file_suffix in zip(glofas_arrays, plot_titles, file_suffixes):
        file_name = f"glofas_usgs_{file_suffix}_{start_forecast}.png"
        file_path = os.path.join(dir_path, file_name)

        if os.path.exists(file_path):
            print(f"File {file_name} already exists.")
            continue

        # Select the corresponding USGS data based on the forecast type
        if file_suffix == 'medium':
            df_filtered = filter_dataframe(usgs_data, start_forecast, 10, 45)
        elif file_suffix == 'seasonal':
            df_filtered = filter_dataframe(usgs_data, start_forecast, 30, 270)
        elif file_suffix == 'reforecast':
            df_filtered = filter_dataframe(usgs_data, start_forecast, 30, 270)
        else:
            raise ValueError("Invalid forecast type.")

        plt.figure(figsize=(20, 6))
        start_forecast_dt = pd.Timestamp(start_forecast)
        lead_times_hours = np.arange(glofas_array.shape[1]) * 24  # Assuming 24-hour intervals
        ensemble_timestamps = start_forecast_dt + pd.to_timedelta(lead_times_hours, unit='h')

        # Annotations for start and end of forecast
        bbox_props = dict(boxstyle="round,pad=0.3", fc="white", ec="pink", lw=2)
        plt.annotate(f'Start: {start_forecast}', (start_forecast_dt, 1), textcoords="offset points", xytext=(0,250), ha='center', bbox=bbox_props)

        # Check if the ensemble_timestamps is not empty
        if len(ensemble_timestamps) > 0:
            end_forecast_dt = ensemble_timestamps[-1]
            plt.annotate(f'End: {end_forecast_dt.strftime("%Y-%m-%d")}', (end_forecast_dt, 1), textcoords="offset points", xytext=(0,250), ha='center', bbox=bbox_props)
            plt.axvline(end_forecast_dt, color='pink', linestyle='--', linewidth=2)

        plt.axvline(start_forecast_dt, color='darkred', linestyle='--', linewidth=2)


        # Plot GloFAS data for each ensemble member
        for i in range(glofas_array.shape[0]):
            plt.plot(ensemble_timestamps, np.log(glofas_array[i, :] + 1), color='orange', alpha=0.2)

        # Plot mean ensemble
        mean_ensemble = np.mean(glofas_array, axis=0)
        plt.plot(ensemble_timestamps, np.log(mean_ensemble + 1), color='darkorange', linewidth=2.5, label='Mean Ensemble (GloFAS)')

        # Plot USGS data
        plt.plot(df_filtered.index, df_filtered['log_discharge_cms'], marker='o', linestyle='--', color='forestgreen', markersize=2, alpha=1, label='USGS Data')

        # Formatting and saving the plot
        plt.xlabel('Time')
        plt.ylabel('Log Discharge (cms)')
        plt.title(plot_title)
        plt.grid(True)
        plt.legend()
        plt.savefig(file_path)

        print(f"Saved the plot as {file_name}")

        # Usage
# glofas_arrays = [glofas_array, glofas_array_s, glofas_array_rf]
# glofas_arrays = [glofas_array, glofas_array_rf]
glofas_arrays = [glofas_array]
plot_titles = ["GloFAS Medium Range Forecast", "GloFAS Reforecast"]
# plot_titles = ["GloFAS Medium Range Forecast", "GloFAS Seasonal Forecast", "GloFAS Reforecast"]
file_suffixes = ["medium", "reforecast"]
# file_suffixes = ["medium", "seasonal", "reforecast"]
plot_glofas_forecasts_with_usgs(glofas_arrays, df, start_forecast, directory_path_figures, plot_titles, file_suffixes)




In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Define the date range for filtering
start_date = '1987-05-29'
end_date = '2022-12-25'

# Filter data to only include the specified date range
filtered_usgs = combined_data_cleaned['USGS'].loc[start_date:end_date]
filtered_glofas = combined_data_cleaned['GloFAS'].loc[start_date:end_date]
filtered_nws3 = combined_data_cleaned['NWS3.0'].loc[start_date:end_date]

# Create a 3x1 matrix plot
fig, axes = plt.subplots(3, 1, figsize=(15, 15), facecolor='white')

# Plot USGS data (First plot)
axes[0].plot(filtered_usgs.index, filtered_usgs, marker='o', linestyle='-', color='green', markersize=1, alpha=0.8)
axes[0].set_xlabel('', fontsize=12, fontweight='bold')
axes[0].set_ylabel('log - Flow Discharge', fontsize=12, fontweight='bold')
axes[0].set_title('USGS - Retrospective', fontsize=14, fontweight='bold')
axes[0].grid(visible=True, linestyle='-', alpha=0.6)

# Plot GloFAS data (Second plot)
axes[1].plot(filtered_glofas.index, filtered_glofas, marker='o', linestyle='-', color='orange', markersize=1, alpha=0.8)
axes[1].set_xlabel('', fontsize=12, fontweight='bold')
axes[1].set_ylabel('log - Flow Discharge', fontsize=12, fontweight='bold')
axes[1].set_title('GloFAS - Retrospective', fontsize=14, fontweight='bold')
axes[1].grid(visible=True, linestyle='-', alpha=0.6)

# Plot NWS3.0 data (Third plot)
axes[2].plot(filtered_nws3.index, filtered_nws3, marker='o', linestyle='-', color='darkred', markersize=1, alpha=0.8)
axes[2].set_xlabel('Date', fontsize=12, fontweight='bold')
axes[2].set_ylabel('log - Flow Discharge', fontsize=12, fontweight='bold')
axes[2].set_title('NWS3.0 - Retrospective', fontsize=14, fontweight='bold')
axes[2].grid(visible=True, linestyle='-', alpha=0.6)

# General layout adjustments
plt.tight_layout(rect=[0, 0, 1, 0.95])

# Save the figure as a high-resolution .png file
output_path = "/data/muscat_data/jaguir26/project1_ucsc_phd/retrospective_comparison.png"
print(f"Saving plot to {output_path}")
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print("Plot saved successfully.")
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Define the date range for filtering
start_date = '1987-05-29'
end_date = '2022-12-25'

# Filter data to only include the specified date range
filtered_usgs = combined_data_cleaned['USGS'].loc[start_date:end_date]
filtered_glofas = combined_data_cleaned['GloFAS'].loc[start_date:end_date]
filtered_nws3 = combined_data_cleaned['NWS3.0'].loc[start_date:end_date]

# Create a 2x1 matrix plot
fig, axes = plt.subplots(2, 1, figsize=(15, 7), facecolor='white')

# Plot GloFAS data (First plot)
axes[0].plot(filtered_glofas.index, filtered_glofas, marker='o', linestyle='-', color='orange', markersize=1, alpha=0.8)
axes[0].set_xlabel('', fontsize=12, fontweight='bold')
axes[0].set_ylabel('log - Flow Discharge', fontsize=12, fontweight='bold')
axes[0].set_title('GloFAS - Retrospective', fontsize=14, fontweight='bold')
axes[0].grid(visible=True, linestyle='-', alpha=0.6)
axes[0].set_ylim(0, 6)  # Set y-axis limits between 0 and 6

# Plot NWS3.0 data (Second plot)
axes[1].plot(filtered_nws3.index, filtered_nws3, marker='o', linestyle='-', color='darkred', markersize=1, alpha=0.8)
axes[1].set_xlabel('Date', fontsize=12, fontweight='bold')
axes[1].set_ylabel('log - Flow Discharge', fontsize=12, fontweight='bold')
axes[1].set_title('NWS3.0 - Retrospective', fontsize=14, fontweight='bold')
axes[1].grid(visible=True, linestyle='-', alpha=0.6)
axes[1].set_ylim(0, 6)  # Set y-axis limits between 0 and 6

# General layout adjustments
plt.tight_layout(rect=[0, 0, 1, 0.95])

# Save the figure as a high-resolution .png file
output_path = "/data/muscat_data/jaguir26/project1_ucsc_phd/retrospective_comparison.png"
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print("Plot saved successfully.")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np

def standardize_values(values):
    """ Standardize the given Pandas Series. """
    mean = values.mean()
    std = values.std()
    return (values - mean) / std

def plot_ensemble_forecasts_and_data(daily_averages, df_usgs, glofas_array, start_forecast, ensemble_list=None, start_date='2018-01-01', end_date=None):
    plt.figure(figsize=(20, 8))
    colormap = plt.cm.cividis
    
    # Define the maximum expected number of ensemble members and generate colors
    max_ensemble = 10
    colors = {num: colormap(i / max_ensemble) for i, num in enumerate(range(1, max_ensemble + 1))}

    # Parse dates if provided as strings
    start_date = pd.to_datetime(start_date)
    if end_date:
        end_date = pd.to_datetime(end_date)

    # Filter data by date range
    if end_date:
        daily_averages = daily_averages.loc[pd.to_datetime('2022-12-26'):pd.to_datetime('2023-01-06')]
        df_usgs = df_usgs.loc[start_date:end_date]
    else:
        daily_averages = daily_averages.loc[pd.to_datetime('2022-12-26'):pd.to_datetime('2023-01-06')]
        df_usgs = df_usgs.loc[start_date:]

    # Create a copy of the DataFrame to avoid SettingWithCopyWarning
    df_usgs_copy = df_usgs.copy()

    # Standardize USGS values - NO NEED OF LOD TRANSF
    usgs_values = df_usgs_copy['log_discharge_cms']
    # usgs_values = standardize_values(usgs_values)
    df_usgs_copy['standardized'] = usgs_values

    # Split USGS data into before and after 2022-12-26
    usgs_before = df_usgs_copy.loc[df_usgs_copy.index < '2022-12-26']
    usgs_after = df_usgs_copy.loc[df_usgs_copy.index >= '2022-12-26']

    # Plot USGS observations before and after 2022-12-26 with different colors and styles
    plt.plot(usgs_before.index, usgs_before['standardized'], label='USGS (Pre 2022-12-26)', linestyle='--', color='green', marker='o', markersize=3)
    plt.plot(usgs_after.index, usgs_after['standardized'], label='USGS (Post 2022-12-26)', linestyle='--', color='lightgreen', marker='o', markersize=6)

    # Standardize and plot NWS3.0 data - NO NEED OF LOD TRANSF
    ft = combined_data_cleaned['NWS3.0'].loc[start_date:pd.to_datetime('2022-12-25')].copy()
    # ft_standardized = standardize_values(np.log1p(ft))
    # ft_standardized = standardize_values((ft))
    ft_standardized = ((ft))
    plt.plot(ft.index, ft_standardized, linestyle='solid', color='darkred', markersize=1, label='NWS')

    # Standardize and plot GloFAS data - NO NEED OF LOD TRANSF
    ft = combined_data_cleaned['GloFAS'].loc[start_date:pd.to_datetime('2022-12-25')].copy()
    # ft_standardized = standardize_values((ft))
    ft_standardized = ((ft))
    plt.plot(ft.index, ft_standardized, label='GloFAS', linestyle='solid', color='darkorange', markersize=1)

    # NO NEED LOG TRANSF
    if ensemble_list is None:
        ensemble_list = daily_averages.columns
    ensemble_data = daily_averages[ensemble_list]
    mean_ensemble = ensemble_data.mean(axis=1)

    for ensemble in sorted(ensemble_list):
        if ensemble in daily_averages.columns:
            plt.plot(daily_averages.index, daily_averages[ensemble], color='lightcoral', marker='.', linestyle='--', alpha=0.5, markersize=3)

    plt.plot(daily_averages.index, mean_ensemble, color='darkred', linewidth=1)

    # Define cutoff dates and add annotations if within the date range
    cutoff_dates = {
        pd.Timestamp("2018-09-17"): "NWS1.0",
        pd.Timestamp("2019-06-19"): "NWS2.0",
        pd.Timestamp("2021-04-20"): "Hourly data",
        pd.Timestamp("2019-11-25"): "NWS2.1",
        pd.Timestamp("2023-01-10"): "SC22' flood",
        pd.Timestamp("2023-09-20"): "NWS3.0",
        pd.Timestamp("2022-12-25"): "Cutoff - 2022-12-25"
    }
    for date, description in cutoff_dates.items():
        if start_date <= date <= (end_date if end_date else date):
            plt.axvline(date, color='black', linestyle='--', linewidth=1)
            plt.text(date, plt.gca().get_ylim()[1], description, horizontalalignment='center', verticalalignment='bottom', rotation=0, color='black')

    # NEED!! LOG TRANSF
    # Plotting the GloFAS ensemble data
    if glofas_array.size != 0: 
        num_ensemble_members = glofas_array.shape[0]
        start_forecast_date = pd.Timestamp(start_forecast)
        lead_times_hours = np.arange(glofas_array.shape[1]) * 24  # Assuming 24-hour intervals
        lead_times_timedelta = pd.to_timedelta(lead_times_hours, unit='h')
        ensemble_timestamps = np.array(start_forecast_date + lead_times_timedelta)
        mean_ensemble = np.mean(glofas_array, axis=0)

        for i in range(num_ensemble_members):
            ensemble_member_data = np.array((glofas_array[i, :]))
            plt.plot(ensemble_timestamps, np.log(ensemble_member_data+1), color='orange', marker='.', linestyle='--', alpha=0.2,  markersize=2)

        plt.plot(ensemble_timestamps, np.log(mean_ensemble+1), color='darkorange', linewidth=1)

    plt.title('')
    plt.xlabel('Date')
    plt.ylabel('Standarized - Log Flow Discharge')
    plt.legend(title='Legend')
    plt.gca().xaxis.set_major_locator(mdates.YearLocator())
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.gca().xaxis.set_minor_locator(mdates.MonthLocator())  # Set minor ticks for months
    plt.gca().xaxis.set_minor_formatter(mdates.DateFormatter('%b'))  # Set minor ticks to show month names
    plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=14))  # Set major ticks for specific days
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))  # Format major ticks to show day, month, and year
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()  

# Example usage:
plot_ensemble_forecasts_and_data(daily_averages, df_usgs, glofas_array, '2022-12-26', ensemble_list=[1, 2, 3, 4, 5, 6, 7], start_date='2022-11-01', end_date='2023-02-01')


In [ ]:
# import matplotlib.pyplot as plt
# import matplotlib.dates as mdates
# import pandas as pd
# import numpy as np
# from matplotlib.animation import FuncAnimation, PillowWriter

# def animate_combined_ensembles(df_usgs, nws_retro, glofas_retro, daily_averages, glofas_array, start_date, end_date, save_path):
#     # Convert indices to timezone-aware if they are not already
#     if df_usgs.index.tz is None:
#         df_usgs.index = df_usgs.index.tz_localize('UTC')
#     if daily_averages.index.tz is None:
#         daily_averages.index = daily_averages.index.tz_localize('UTC')
    
#     fig, ax = plt.subplots(figsize=(20, 8), dpi=300)  # High DPI for better quality

#     # Parse dates if provided as strings
#     start_date = pd.to_datetime(start_date).tz_localize('UTC')
#     end_date = pd.to_datetime(end_date).tz_localize('UTC') if end_date else df_usgs.index[-1]

#     # Filter data by date range
#     df_usgs = df_usgs.loc[start_date:end_date]
#     nws_retro = nws_retro.loc[start_date:pd.to_datetime('2022-12-25').tz_localize('UTC')]
#     glofas_retro = glofas_retro.loc[start_date:pd.to_datetime('2022-12-25').tz_localize('UTC')]

#     # Split USGS data into before and after 2022-12-26
#     usgs_before_cutoff = df_usgs.loc[df_usgs.index <= '2022-12-26']
#     usgs_after_cutoff = df_usgs.loc[df_usgs.index >= '2022-12-26']

#     # Prepare data for the given date range
#     daily_averages_selected = daily_averages.loc[pd.to_datetime('2022-12-26').tz_localize('UTC'):pd.to_datetime('2023-01-06').tz_localize('UTC')]
#     glofas_array_selected = glofas_array[:, :len(daily_averages_selected)]

#     # Log transform the GloFAS ensemble data
#     glofas_array_log = np.log1p(glofas_array_selected)

#     # Prepare lines for animation
#     line_usgs, = ax.plot([], [], label='USGS (Pre 2022-12-26)', linestyle='--', color='green', marker='o', markersize=3)
#     line_usgs_after, = ax.plot([], [], label='USGS (Post 2022-12-26)', linestyle='--', color='lightgreen', marker='o', markersize=6)
#     line_nws, = ax.plot([], [], linestyle='solid', color='darkred', markersize=1, label='NWS')
#     line_glofas, = ax.plot([], [], linestyle='solid', color='darkorange', markersize=1, label='GloFAS')

#     nws_ensemble_lines = [ax.plot([], [], linestyle='--', color='lightcoral', alpha=0.5)[0] for _ in range(daily_averages_selected.shape[1])]
#     glofas_ensemble_lines = [ax.plot([], [], linestyle='--', color='orange', alpha=0.2)[0] for _ in range(glofas_array_selected.shape[0])]
#     mean_nws_line, = ax.plot([], [], color='darkred', linewidth=1)
#     mean_glofas_line, = ax.plot([], [], color='darkorange', linewidth=1)

#     def init():
#         ax.set_xlim(start_date, end_date)
#         ax.set_ylim(-1.2, 6)
#         ax.set_xlabel('Date')
#         ax.set_ylabel('Log Flow Discharge')
#         ax.legend(title='Legend')
#         ax.xaxis.set_major_locator(mdates.YearLocator())
#         ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
#         ax.xaxis.set_minor_locator(mdates.MonthLocator())
#         ax.xaxis.set_minor_formatter(mdates.DateFormatter('%b'))
#         ax.xaxis.set_major_locator(mdates.DayLocator(interval=14))  # Add major ticks for specific days
#         ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))  # Format major ticks to show day, month, and year
#         ax.grid(True)
#         plt.xticks(rotation=45)
#         plt.tight_layout()
        
#         # Add vertical lines and annotations
#         cutoff_dates = {
#             pd.Timestamp("2018-09-17").tz_localize('UTC'): "NWS1.0",
#             pd.Timestamp("2019-06-19").tz_localize('UTC'): "NWS2.0",
#             pd.Timestamp("2021-04-20").tz_localize('UTC'): "Hourly data",
#             pd.Timestamp("2019-11-25").tz_localize('UTC'): "NWS2.1",
#             pd.Timestamp("2023-01-10").tz_localize('UTC'): "SC22' flood",
#             pd.Timestamp("2023-09-20").tz_localize('UTC'): "NWS3.0",
#             pd.Timestamp("2022-12-25").tz_localize('UTC'): "Cutoff - 2022-12-25"
#         }
#         for date, description in cutoff_dates.items():
#             if start_date <= date <= end_date:
#                 ax.axvline(date, color='black', linestyle='--', linewidth=1)
#                 ax.text(date, ax.get_ylim()[1], description, horizontalalignment='center', verticalalignment='bottom', rotation=0, color='black')

#         return [line_usgs, line_usgs_after, line_nws, line_glofas] + nws_ensemble_lines + glofas_ensemble_lines + [mean_nws_line, mean_glofas_line]

#     def update(frame):
#         if frame < len(usgs_before_cutoff):
#             # Update USGS data before the cutoff
#             line_usgs.set_data(usgs_before_cutoff.index[:frame], usgs_before_cutoff['log_discharge_cms'][:frame])
#         elif frame < len(usgs_before_cutoff) + len(nws_retro):
#             reverse_frame = frame - len(usgs_before_cutoff) + 1
#             # Update NWS data in reverse
#             line_nws.set_data(nws_retro.index[-reverse_frame:], nws_retro[-reverse_frame:])
#             # Update GloFAS data in reverse
#             line_glofas.set_data(glofas_retro.index[-reverse_frame:], glofas_retro[-reverse_frame:])
#         elif frame < len(usgs_before_cutoff) + len(nws_retro) + len(daily_averages_selected):
#             ensemble_frame = frame - len(usgs_before_cutoff) - len(nws_retro) + 1
#             date_range = pd.date_range(start=pd.to_datetime('2022-12-26').tz_localize('UTC'), periods=ensemble_frame, freq='D')
#             for i, line in enumerate(nws_ensemble_lines):
#                 line.set_data(date_range, daily_averages_selected.iloc[:ensemble_frame, i])
#             mean_nws_line.set_data(date_range, daily_averages_selected.mean(axis=1)[:ensemble_frame])
#             for j, line in enumerate(glofas_ensemble_lines):
#                 line.set_data(date_range, glofas_array_log[j, :ensemble_frame])
#             mean_glofas_line.set_data(date_range, glofas_array_log.mean(axis=0)[:ensemble_frame])
#         else:
#             usgs_after_frame = frame - len(usgs_before_cutoff) - len(nws_retro) - len(daily_averages_selected) + 1
#             line_usgs_after.set_data(usgs_after_cutoff.index[:usgs_after_frame], usgs_after_cutoff['log_discharge_cms'][:usgs_after_frame])
#         return [line_usgs, line_usgs_after, line_nws, line_glofas] + nws_ensemble_lines + glofas_ensemble_lines + [mean_nws_line, mean_glofas_line]

#     total_frames = len(usgs_before_cutoff) + len(nws_retro) + len(daily_averages_selected) + len(usgs_after_cutoff)
#     anim = FuncAnimation(fig, update, frames=np.arange(1, total_frames + 1), init_func=init, blit=True)

#     # Save the animation as GIF with high quality settings
#     anim.save(save_path, writer=PillowWriter(fps=10))

#     plt.show()

#     return anim

# # Example usage:
# combined_data_cleaned2 = pd.read_csv('/data/muscat_data/jaguir26/project1_ucsc_phd/combined_streamflow_data_cleaned.csv', parse_dates=['Date'], index_col='Date')
# combined_data_cleaned2.index = combined_data_cleaned2.index.tz_localize('UTC')
# nws_retro = combined_data_cleaned2['NWS3.0'].copy()
# glofas_retro = combined_data_cleaned2['GloFAS'].copy()

# daily_averages2 = daily_averages.copy()
# daily_averages2.index = daily_averages2.index.tz_localize('UTC')
# daily_averages2 = daily_averages2.loc[pd.to_datetime('2022-12-26').tz_localize('UTC'):pd.to_datetime('2023-01-06').tz_localize('UTC')]

# anim = animate_combined_ensembles(df_usgs, nws_retro, glofas_retro, daily_averages2, glofas_array, start_date='2022-11-20', end_date='2023-02-01', save_path='/data/muscat_data/jaguir26/project1_ucsc_phd/combined_ensemble_animation.gif')


## Data + Forecasts
- Have all data on the same units
- Take the logarithm + 1
- Standarize just at the end (how)?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta

# Constants
CFSToCMS_CONVERSION_FACTOR = 0.0283168466
site_code = '11160500'
start_date = '2022-11-01'
end_date = '2023-01-31'

# USGS Data Processing
def process_usgs_data():
    start_usgs = '1979-01-01'
    today = datetime.today()
    end_usgs = today.strftime('%Y-%m-%d')
    df = nwis.get_record(sites=site_code, service='dv', 
                         parameterCd='00060', 
                         statCd='00003',
                         start=start_usgs, 
                         end=end_usgs)
    df['discharge_cms'] = df['00060_Mean'].astype(float) * CFSToCMS_CONVERSION_FACTOR
    df['log_streamflow'] = np.log1p(df['discharge_cms'])
    df['Date'] = pd.to_datetime(df.index).date
    daily_avg_usgs = df.groupby('Date').agg(Daily_Avg_Log_Streamflow=('log_streamflow', 'mean')).reset_index()
    daily_avg_usgs['Date'] = pd.to_datetime(daily_avg_usgs['Date'])  # Ensure Date is in Timestamp format
    return daily_avg_usgs

# GLOFAS Retro Data Processing
def process_glofas_data():
    project_input_dir = "/data/muscat_data/jaguir26/projects/Project/Input/Retrospective_Analysis/GLOFAS"
    glofas_path = os.path.join(project_input_dir, 'glofas_1979_2023', 'glofas_streamflow_data.csv')
    df_glofas = pd.read_csv(glofas_path)
    df_glofas['Date'] = pd.to_datetime(df_glofas['Date']).dt.date
    df_glofas['log_streamflow'] = np.log1p(df_glofas['Streamflow'])
    daily_avg_glofas = df_glofas.groupby('Date').agg(Daily_Avg_Log_Streamflow=('log_streamflow', 'mean')).reset_index()
    daily_avg_glofas['Date'] = pd.to_datetime(daily_avg_glofas['Date'])  # Ensure Date is in Timestamp format
    return daily_avg_glofas

# NWS Retro Data Processing
def process_nws_data():
    dir = "/data/muscat_data/jaguir26/project1_ucsc_phd"
    nwm_new_path = os.path.join(dir, f'{site_code}_nws_retro.csv')
    df_nwm_new = pd.read_csv(nwm_new_path)
    df_nwm_new['Date'] = pd.to_datetime(df_nwm_new['Date']).dt.date
    df_nwm_new['log_streamflow'] = np.log1p(df_nwm_new['streamflow'])
    daily_avg_nws = df_nwm_new.groupby('Date').agg(Daily_Avg_Log_Streamflow=('log_streamflow', 'mean')).reset_index()
    daily_avg_nws['Date'] = pd.to_datetime(daily_avg_nws['Date'])  # Ensure Date is in Timestamp format
    return daily_avg_nws

# GLOFAS Ensemble Data Processing
def process_glofas_ensemble(start_forecast, glofas_array):
    num_ensemble_members = glofas_array.shape[0]
    start_forecast_date = pd.Timestamp(start_forecast)
    lead_times_hours = np.arange(glofas_array.shape[1]) * 24  # Assuming 24-hour intervals
    lead_times_timedelta = pd.to_timedelta(lead_times_hours, unit='h')
    ensemble_timestamps = np.array(start_forecast_date + lead_times_timedelta)
    
    glofas_ens_df = pd.DataFrame(glofas_array.T, index=ensemble_timestamps, columns=[f'Ensemble_Member_{i+1}' for i in range(num_ensemble_members)])
    glofas_ens_df = glofas_ens_df.apply(np.vectorize(np.log1p))

    # Convert index to date for daily averaging
    glofas_ens_df['Date'] = glofas_ens_df.index.date
    daily_avg_glofas_ens = glofas_ens_df.groupby('Date').mean().reset_index()
    daily_avg_glofas_ens['Date'] = pd.to_datetime(daily_avg_glofas_ens['Date'])  # Ensure Date is in Timestamp format
    return daily_avg_glofas_ens

# # NWS Forecast Data Processing
# def process_nws_forecast(pkl_file_path):
#     forecast_df = extract_forecast_data(pkl_file_path)
#     forecast_df['Target_Time'] = forecast_df['Date'] + pd.to_timedelta(forecast_df['Lead_Time'], unit='h')
#     forecast_df['Weight'] = 1 / forecast_df['Lead_Time']
#     forecast_df['Normalized_Weight'] = forecast_df.groupby(['Target_Time', 'Ensemble_Number'])['Weight'].transform(lambda x: x / x.sum())
#     forecast_df['Transformed_Value'] = np.log1p(forecast_df['Value'])
    
#     weighted_avg_df = forecast_df.groupby(['Target_Time', 'Ensemble_Number']).apply(
#         lambda x: np.sum(x['Transformed_Value'] * x['Normalized_Weight'])
#     ).reset_index(name='Weighted_Avg_Transformed_Value')
    
#     # Convert 'Target_Time' to just the date for daily averaging
#     weighted_avg_df['Date'] = weighted_avg_df['Target_Time'].dt.date
#     daily_avg_df = weighted_avg_df.groupby(['Date', 'Ensemble_Number']).agg(
#         Daily_Avg_Transformed_Value=('Weighted_Avg_Transformed_Value', 'mean')
#     ).reset_index()
#     daily_avg_df['Date'] = pd.to_datetime(daily_avg_df['Date'])  # Ensure Date is in Timestamp format
#     return daily_avg_df

cutoff_date = '2022-12-25'  # Forecasts produced on or before this date

# NWS Forecast Data Processing
def process_nws_forecast(pkl_file_path, cutoff_date):
    forecast_df = extract_forecast_data(pkl_file_path)
    cutoff_date = pd.to_datetime(cutoff_date)
    
    # Filter the forecasts based on the cutoff date
    forecast_df = forecast_df[forecast_df['Date'] <= cutoff_date]
    
    forecast_df['Target_Time'] = forecast_df['Date'] + pd.to_timedelta(forecast_df['Lead_Time'], unit='h')
    forecast_df['Weight'] = 1 / forecast_df['Lead_Time']
    forecast_df['Normalized_Weight'] = forecast_df.groupby(['Target_Time', 'Ensemble_Number'])['Weight'].transform(lambda x: x / x.sum())
    forecast_df['Transformed_Value'] = np.log1p(forecast_df['Value'])
    
    weighted_avg_df = forecast_df.groupby(['Target_Time', 'Ensemble_Number']).apply(
        lambda x: np.sum(x['Transformed_Value'] * x['Normalized_Weight'])
    ).reset_index(name='Weighted_Avg_Transformed_Value')
    
    # Convert 'Target_Time' to just the date for daily averaging
    weighted_avg_df['Date'] = weighted_avg_df['Target_Time'].dt.date
    
    # Filter the target times to ensure the maximum target date is within the required range
    max_target_date = (cutoff_date + pd.to_timedelta(forecast_df['Lead_Time'].max(), unit='h')).date()
    weighted_avg_df = weighted_avg_df[weighted_avg_df['Date'] <= max_target_date]
    
    daily_avg_df = weighted_avg_df.groupby(['Date', 'Ensemble_Number']).agg(
        Daily_Avg_Transformed_Value=('Weighted_Avg_Transformed_Value', 'mean')
    ).reset_index()
    daily_avg_df['Date'] = pd.to_datetime(daily_avg_df['Date'])  # Ensure Date is in Timestamp format
    
    return daily_avg_df


In [ ]:

# Process all datasets
daily_avg_usgs = process_usgs_data()
daily_avg_glofas = process_glofas_data()
daily_avg_nws = process_nws_data()
start_forecast = '2022-12-26'
daily_avg_glofas_ens = process_glofas_ensemble(start_forecast, glofas_array)
pkl_file_path = '/data/muscat_data/jaguir26/project1_ucsc_phd/results.pkl'
# daily_avg_nws_forecast = process_nws_forecast(pkl_file_path)
pkl_file_path = '/data/muscat_data/jaguir26/project1_ucsc_phd/results.pkl'
daily_avg_nws_forecast = process_nws_forecast(pkl_file_path, cutoff_date)


In [ ]:
cutoff_date = '2022-12-25'
pkl_file_path = '/data/muscat_data/jaguir26/project1_ucsc_phd/results.pkl'
forecast_df = extract_forecast_data(pkl_file_path)
cutoff_date = pd.to_datetime(cutoff_date)
forecast_df = forecast_df[forecast_df['Date'] <= cutoff_date]

In [ ]:
forecast_df
# forecast_df[forecast_df['Date']>='2022-12-24']

In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np
# import pandas as pd
# from datetime import timedelta
# import matplotlib.dates as mdates
# import pickle
# from datetime import datetime

# def extract_forecast_data(filepath):
#     with open(filepath, 'rb') as file:
#         data = pickle.load(file)
#     forecast_entries = []
#     for key, value in data.items():
#         parts = key.split('/')
#         date_part = parts[0].split('.')[1]
#         forecast_date = datetime.strptime(date_part, '%Y%m%d')
#         ensemble_part = parts[1]
#         ensemble_number = int(ensemble_part.split('mem')[1]) if 'mem' in ensemble_part else 1
#         lead_time = int(parts[2].split('f')[1].split('.')[0])
#         forecast_entries.append((forecast_date, ensemble_number, lead_time, value))
#     return pd.DataFrame(forecast_entries, columns=['Date', 'Ensemble_Number', 'Lead_Time', 'Value'])

# def calculate_weight(ensemble_number, lead_time):
#     exponent_dict = {
#         1: 0,
#         2: 0.3,
#         3: 0.6,
#         4: 0.9,
#         5: 1.2,
#         6: 1.5,
#         7: 1.8
#     }
#     return 1 / (lead_time ** exponent_dict.get(ensemble_number, 1))

# # NWS Forecast Data Processing
# def process_nws_forecast(pkl_file_path):
#     forecast_df = extract_forecast_data(pkl_file_path)
#     forecast_df['Target_Time'] = forecast_df['Date'] + pd.to_timedelta(forecast_df['Lead_Time'], unit='h')
    
#     # Apply different weighting methods
#     forecast_df['Weight'] = forecast_df.apply(lambda row: calculate_weight(row['Ensemble_Number'], row['Lead_Time']), axis=1)
#     forecast_df['Normalized_Weight'] = forecast_df.groupby(['Target_Time', 'Ensemble_Number'])['Weight'].transform(lambda x: x / x.sum())
    
#     # Transform the values
#     forecast_df['Transformed_Value'] = np.log1p(forecast_df['Value'])
    
#     weighted_avg_df = forecast_df.groupby(['Target_Time', 'Ensemble_Number']).apply(
#         lambda x: np.sum(x['Transformed_Value'] * x['Normalized_Weight'])
#     ).reset_index(name='Weighted_Avg_Transformed_Value')
    
#     # Convert 'Target_Time' to just the date for daily averaging
#     weighted_avg_df['Date'] = weighted_avg_df['Target_Time'].dt.date
#     daily_avg_df = weighted_avg_df.groupby(['Date', 'Ensemble_Number']).agg(
#         Daily_Avg_Transformed_Value=('Weighted_Avg_Transformed_Value', 'mean')
#     ).reset_index()
#     daily_avg_df['Date'] = pd.to_datetime(daily_avg_df['Date'])  # Ensure Date is in Timestamp format
#     return daily_avg_df

# # Process all datasets

# pkl_file_path = '/data/muscat_data/jaguir26/project1_ucsc_phd/results.pkl'
# daily_avg_nws_forecast_dif_w = process_nws_forecast(pkl_file_path)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datetime import timedelta, datetime
import matplotlib.dates as mdates
import pickle

def extract_forecast_data(filepath):
    with open(filepath, 'rb') as file:
        data = pickle.load(file)
    forecast_entries = []
    for key, value in data.items():
        parts = key.split('/')
        date_part = parts[0].split('.')[1]
        forecast_date = datetime.strptime(date_part, '%Y%m%d')
        ensemble_part = parts[1]
        ensemble_number = int(ensemble_part.split('mem')[1]) if 'mem' in ensemble_part else 1
        lead_time = int(parts[2].split('f')[1].split('.')[0])
        forecast_entries.append((forecast_date, ensemble_number, lead_time, value))
    return pd.DataFrame(forecast_entries, columns=['Date', 'Ensemble_Number', 'Lead_Time', 'Value'])

def calculate_weight(ensemble_number, lead_time):
    exponent_dict = {
        1: 0,
        2: 0.3,
        3: 0.6,
        4: 0.9,
        5: 1.2,
        6: 1.5,
        7: 1.8
    }
    return 1 / (lead_time ** exponent_dict.get(ensemble_number, 1))

# NWS Forecast Data Processing
def process_nws_forecast(pkl_file_path, cutoff_date):
    forecast_df = extract_forecast_data(pkl_file_path)
    cutoff_date = pd.to_datetime(cutoff_date)
    
    # Filter the forecasts based on the cutoff date
    forecast_df = forecast_df[forecast_df['Date'] <= cutoff_date]
    
    forecast_df['Target_Time'] = forecast_df['Date'] + pd.to_timedelta(forecast_df['Lead_Time'], unit='h')
    
    # Apply different weighting methods
    forecast_df['Weight'] = forecast_df.apply(lambda row: calculate_weight(row['Ensemble_Number'], row['Lead_Time']), axis=1)
    forecast_df['Normalized_Weight'] = forecast_df.groupby(['Target_Time', 'Ensemble_Number'])['Weight'].transform(lambda x: x / x.sum())
    
    # Transform the values
    forecast_df['Transformed_Value'] = np.log1p(forecast_df['Value'])
    
    weighted_avg_df = forecast_df.groupby(['Target_Time', 'Ensemble_Number']).apply(
        lambda x: np.sum(x['Transformed_Value'] * x['Normalized_Weight'])
    ).reset_index(name='Weighted_Avg_Transformed_Value')
    
    # Convert 'Target_Time' to just the date for daily averaging
    weighted_avg_df['Date'] = weighted_avg_df['Target_Time'].dt.date
    
    # Filter the target times to ensure the maximum target date is within the required range
    max_target_date = (cutoff_date + pd.to_timedelta(forecast_df['Lead_Time'].max(), unit='h')).date()
    weighted_avg_df = weighted_avg_df[weighted_avg_df['Date'] <= max_target_date]
    
    daily_avg_df = weighted_avg_df.groupby(['Date', 'Ensemble_Number']).agg(
        Daily_Avg_Transformed_Value=('Weighted_Avg_Transformed_Value', 'mean')
    ).reset_index()
    daily_avg_df['Date'] = pd.to_datetime(daily_avg_df['Date'])  # Ensure Date is in Timestamp format
    
    return daily_avg_df

# Example usage
pkl_file_path = '/data/muscat_data/jaguir26/project1_ucsc_phd/results.pkl'
cutoff_date = '2022-12-25'
daily_avg_nws_forecast_dif_w = process_nws_forecast(pkl_file_path, cutoff_date)

# Print the result to verify
print(daily_avg_nws_forecast_dif_w)

In [ ]:
def plot_daily_avg_forecast(daily_avg_df):
    # Filter data between December 2022 and February 2023
    start_date = pd.to_datetime('2022-12-01')
    end_date = pd.to_datetime('2023-01-01')
    filtered_df = daily_avg_df[(daily_avg_df['Date'] >= start_date) & (daily_avg_df['Date'] <= end_date)]
    
    plt.figure(figsize=(14, 8))
    for ensemble in filtered_df['Ensemble_Number'].unique():
        ensemble_data = filtered_df[filtered_df['Ensemble_Number'] == ensemble]
        plt.plot(ensemble_data['Date'], ensemble_data['Daily_Avg_Transformed_Value'], label=f'Ensemble {ensemble}')
    
    plt.xlabel('Date')
    plt.ylabel('Daily Average Transformed Value')
    plt.title('Daily Average Transformed Value by Ensemble Member (Dec 2022 - Feb 2023)')
    plt.legend()
    plt.grid(True)
    plt.show()

plot_daily_avg_forecast(daily_avg_nws_forecast_dif_w)

In [ ]:
daily_avg_nws_forecast.to_csv('/data/muscat_data/jaguir26/project1_ucsc_phd/nws_daily_avg_forecast.csv', index=False)
daily_avg_nws_forecast_dif_w.to_csv('/data/muscat_data/jaguir26/project1_ucsc_phd/nws_daily_avg_forecast_dif_w.csv', index=False)
daily_avg_glofas_ens.to_csv('/data/muscat_data/jaguir26/project1_ucsc_phd/glofas_daily_avg_forecast.csv', index=False)

daily_avg_nws.to_csv('/data/muscat_data/jaguir26/project1_ucsc_phd/nws_daily_avg.csv', index=False)
daily_avg_glofas.to_csv('/data/muscat_data/jaguir26/project1_ucsc_phd/glofas_daily_avg.csv', index=False)
daily_avg_usgs.to_csv('/data/muscat_data/jaguir26/project1_ucsc_phd/usgs_daily_avg.csv', index=False)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np

# Load the weighted time series for GloFAS
weighted_time_series_path = "/data/muscat_data/jaguir26/project1_ucsc_phd/weighted_time_series.csv"
weighted_time_series = pd.read_csv(weighted_time_series_path)
weighted_time_series['target_date'] = pd.to_datetime(weighted_time_series['target_date'])

def plot_ensemble_forecasts_and_data(daily_averages, df_usgs, start_forecast, end_forecast, daily_avg_nws_forecast_dif_w, weighted_time_series, start_date='2018-01-01', end_date=None):
    plt.figure(figsize=(20, 8))
    colormap = plt.cm.cividis
    
    max_ensemble = 10
    colors = {num: colormap(i / max_ensemble) for i, num in enumerate(range(1, max_ensemble + 1))}

    start_date = pd.to_datetime(start_date)
    if end_date:
        end_date = pd.to_datetime(end_date)

    if end_date:
        daily_averages = daily_averages.loc[pd.to_datetime(start_forecast):(pd.to_datetime(start_forecast)+ pd.Timedelta(days=10))]
        df_usgs = df_usgs.loc[start_date:end_date]
    else:
        daily_averages = daily_averages.loc[pd.to_datetime(start_forecast):(pd.to_datetime(start_forecast)+ pd.Timedelta(days=10))]
        df_usgs = df_usgs.loc[start_date:]

    df_usgs_copy = df_usgs.copy()

    usgs_values = df_usgs_copy['log_discharge_cms']
    df_usgs_copy['standardized'] = usgs_values

    # Split USGS data into before and after start_forecast
    usgs_before = df_usgs_copy.loc[df_usgs_copy.index < start_forecast]
    usgs_after = df_usgs_copy.loc[df_usgs_copy.index >= start_forecast]

    # Plot USGS observations before and after start_forecast with different colors and styles
    plt.plot(usgs_before.index, usgs_before['standardized'], label=f'USGS (Pre {start_forecast})', linestyle='--', color='green', marker='o', markersize=3)
    plt.plot(usgs_after.index, usgs_after['standardized'], label=f'USGS (Post {start_forecast})', linestyle='--', color='lightgreen', marker='o', markersize=6)

    ft = combined_data_cleaned['NWS3.0'].loc[start_date:pd.to_datetime(start_forecast)].copy()
    ft_standardized = ((ft))
    plt.plot(ft.index, ft_standardized, linestyle='solid', color='darkred', markersize=1, label='NWS')

    ft = combined_data_cleaned['GloFAS'].loc[start_date:pd.to_datetime(start_forecast)].copy()
    ft_standardized = ((ft))
    plt.plot(ft.index, ft_standardized, label='GloFAS', linestyle='solid', color='darkorange', markersize=1)
    
    ensemble_list = daily_averages.columns
    ensemble_data = daily_averages[ensemble_list]
    mean_ensemble = ensemble_data.mean(axis=1)

    for ensemble in sorted(ensemble_list):
        if ensemble in daily_averages.columns:
            plt.plot(daily_averages.index, daily_averages[ensemble], color='lightcoral', linestyle='-', alpha=0.5, markersize=3)

    plt.plot(daily_averages.index, mean_ensemble, color='darkred', linewidth=0.5, label='Mean Ensemble')

    # Plot the GloFAS weighted ensembles for the specified date range
    weighted_time_series_filtered = weighted_time_series[(weighted_time_series['target_date'] >= start_forecast) & (weighted_time_series['target_date'] <= end_forecast)]
    for col in weighted_time_series.columns[1:]:
        plt.plot(weighted_time_series_filtered['target_date'], weighted_time_series_filtered[col], linestyle='-', alpha=0.1, color='orange')
    glofas_mean_ensemble = weighted_time_series_filtered.iloc[:, 1:].mean(axis=1)

    # # Create the combined mean series
    # common_index = weighted_time_series_filtered['target_date']
    # mean_ensemble_reindexed = mean_ensemble.reindex(common_index).values
    # combined_mean = 0.5*glofas_mean_ensemble.values + 0.5*mean_ensemble_reindexed

    # plt.plot(common_index, combined_mean, color='darkblue', linewidth=1, label='Combined Mean (GloFAS & NWS Mean)')

    # Define cutoff dates and add annotations if within the date range
    cutoff_dates = {
        pd.Timestamp("2018-09-17"): "NWS1.0",
        pd.Timestamp("2019-06-19"): "NWS2.0",
        pd.Timestamp("2021-04-20"): "Hourly data",
        pd.Timestamp("2019-11-25"): "NWS2.1",
        pd.Timestamp("2023-01-10"): "SC22' flood",
        pd.Timestamp("2023-09-20"): "NWS3.0",
        pd.Timestamp("2022-12-25"): "Start Forecast - 2022-12-25"
    }
    for date, description in cutoff_dates.items():
        if start_date <= date <= (end_date if end_date else date):
            plt.axvline(date, color='black', linestyle='--', linewidth=1)
            plt.text(date, plt.gca().get_ylim()[1], description, horizontalalignment='center', verticalalignment='bottom', rotation=0, color='black')

    plt.title('')
    plt.xlabel('Date')
    plt.ylabel('Log Flow Discharge')
    plt.legend(title='Legend', loc='upper left')
    plt.gca().xaxis.set_major_locator(mdates.YearLocator())
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.gca().xaxis.set_minor_locator(mdates.MonthLocator())  # Set minor ticks for months
    plt.gca().xaxis.set_minor_formatter(mdates.DateFormatter('%b'))  # Set minor ticks to show month names
    plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=14))  # Set major ticks for specific days
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))  # Format major ticks to show day, month, and year
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()  

# Example usage:
plot_ensemble_forecasts_and_data(daily_averages, df_usgs, '2022-12-26', '2023-01-25', daily_avg_nws_forecast_dif_w, weighted_time_series, start_date='2022-11-01', end_date='2023-02-01')


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np

# Load the weighted time series for GloFAS
weighted_time_series_path = "/data/muscat_data/jaguir26/project1_ucsc_phd/weighted_time_series.csv"
weighted_time_series = pd.read_csv(weighted_time_series_path)
weighted_time_series['target_date'] = pd.to_datetime(weighted_time_series['target_date'])

def plot_ensemble_forecasts_and_data(daily_averages, df_usgs, start_forecast, end_forecast, daily_avg_nws_forecast_dif_w, weighted_time_series, start_date='2018-01-01', end_date=None, save_path=None, plot_glofas_ensembles=True, plot_nws_ensembles=True, plot_glofas_retro=True, plot_nws_retro=True):
    plt.figure(figsize=(20, 8))
    colormap = plt.cm.cividis
    
    max_ensemble = 10
    colors = {num: colormap(i / max_ensemble) for i, num in enumerate(range(1, max_ensemble + 1))}

    start_date = pd.to_datetime(start_date)
    if end_date:
        end_date = pd.to_datetime(end_date)

    if end_date:
        daily_averages = daily_averages.loc[pd.to_datetime(start_forecast):(pd.to_datetime(start_forecast) + pd.Timedelta(days=10))]
        df_usgs = df_usgs.loc[start_date:end_date]
    else:
        daily_averages = daily_averages.loc[pd.to_datetime(start_forecast):(pd.to_datetime(start_forecast) + pd.Timedelta(days=10))]
        df_usgs = df_usgs.loc[start_date:]

    df_usgs_copy = df_usgs.copy()

    usgs_values = df_usgs_copy['log_discharge_cms']
    df_usgs_copy['standardized'] = usgs_values

    # Split USGS data into before and after start_forecast
    usgs_before = df_usgs_copy.loc[df_usgs_copy.index < start_forecast]
    usgs_after = df_usgs_copy.loc[df_usgs_copy.index >= start_forecast]

    # Plot USGS observations before and after start_forecast with different colors and styles
    plt.plot(usgs_before.index, usgs_before['standardized'], label=f'USGS (Pre {start_forecast})', linestyle='--', color='green', marker='o', markersize=3)
    plt.plot(usgs_after.index, usgs_after['standardized'], label=f'USGS (Post {start_forecast})', linestyle='--', color='lightgreen', marker='o', markersize=6)

    if plot_nws_retro:
        ft = combined_data_cleaned['NWS3.0'].loc[start_date:pd.to_datetime(start_forecast)].copy()
        ft_standardized = ((ft))
        plt.plot(ft.index, ft_standardized, linestyle='solid', color='darkblue', markersize=1, label='NWS')

    if plot_glofas_retro:
        ft = combined_data_cleaned['GloFAS'].loc[start_date:pd.to_datetime(start_forecast)].copy()
        ft_standardized = ((ft))
        plt.plot(ft.index, ft_standardized, label='GloFAS', linestyle='solid', color='darkorange', markersize=1)
    
    if plot_nws_ensembles:
        ensemble_list = daily_averages.columns
        ensemble_data = daily_averages[ensemble_list]
        mean_ensemble = ensemble_data.mean(axis=1)

        for ensemble in sorted(ensemble_list):
            if ensemble in daily_averages.columns:
                plt.plot(daily_averages.index, daily_averages[ensemble], color='lightblue', linestyle='-', alpha=0.5, markersize=3)

        plt.plot(daily_averages.index, mean_ensemble, color='darkblue', linewidth=0.5, label='Mean Ensemble')

    if plot_glofas_ensembles:
        # Plot the GloFAS weighted ensembles for the specified date range
        weighted_time_series_filtered = weighted_time_series[(weighted_time_series['target_date'] >= start_forecast) & (weighted_time_series['target_date'] <= (pd.to_datetime(start_forecast) + pd.Timedelta(days=30)))]
        for col in weighted_time_series.columns[1:]:
            plt.plot(weighted_time_series_filtered['target_date'], weighted_time_series_filtered[col], linestyle='-', alpha=0.1, color='orange')
        glofas_mean_ensemble = weighted_time_series_filtered.iloc[:, 1:].mean(axis=1)
        plt.plot(weighted_time_series_filtered['target_date'], glofas_mean_ensemble, color='darkorange', linewidth=1, label='GloFAS Mean')

    # Define cutoff dates and add annotations if within the date range
    cutoff_dates = {
        pd.Timestamp("2018-09-17"): "NWS1.0",
        pd.Timestamp("2019-06-19"): "NWS2.0",
        pd.Timestamp("2021-04-20"): "Hourly data",
        pd.Timestamp("2019-11-25"): "NWS2.1",
        pd.Timestamp("2023-01-10"): "SC22' flood",
        pd.Timestamp("2023-09-20"): "NWS3.0",
        pd.Timestamp(start_forecast): f"Start Forecast - {start_forecast}"
    }
    for date, description in cutoff_dates.items():
        if start_date <= date <= (end_date if end_date else date):
            plt.axvline(date, color='black', linestyle='--', linewidth=1)
            plt.text(date, plt.gca().get_ylim()[1], description, horizontalalignment='center', verticalalignment='bottom', rotation=0, color='black')

    plt.title('')
    plt.xlabel('Date')
    plt.ylabel('Log Flow Discharge')
    plt.legend(title='Legend', loc='upper left')
    plt.gca().xaxis.set_major_locator(mdates.YearLocator())
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.gca().xaxis.set_minor_locator(mdates.MonthLocator())  # Set minor ticks for months
    plt.gca().xaxis.set_minor_formatter(mdates.DateFormatter('%b'))  # Set minor ticks to show month names
    plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=14))  # Set major ticks for specific days
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))  # Format major ticks to show day, month, and year
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300)
    
    plt.show()  


In [ ]:

# Save the plots
plot_ensemble_forecasts_and_data(daily_averages, df_usgs, '2022-12-26', '2023-01-25', daily_avg_nws_forecast_dif_w, weighted_time_series, start_date='2022-11-01', end_date='2023-02-01', save_path='/data/muscat_data/jaguir26/project1_ucsc_phd/plot_with_ensembles.png')

plot_ensemble_forecasts_and_data(daily_averages, df_usgs, '2022-12-26', '2023-01-25', daily_avg_nws_forecast_dif_w, weighted_time_series, start_date='2022-11-01', end_date='2023-02-01', save_path='/data/muscat_data/jaguir26/project1_ucsc_phd/plot_without_ensembles.png', plot_glofas_ensembles=False, plot_nws_ensembles=False)

plot_ensemble_forecasts_and_data(daily_averages, df_usgs, '2022-12-26', '2023-01-25', daily_avg_nws_forecast_dif_w, weighted_time_series, start_date='2022-11-01', end_date='2023-02-01', save_path='/data/muscat_data/jaguir26/project1_ucsc_phd/plot_without_glofas_ensembles.png', plot_glofas_ensembles=False)

plot_ensemble_forecasts_and_data(daily_averages, df_usgs, '2022-12-26', '2023-01-25', daily_avg_nws_forecast_dif_w, weighted_time_series, start_date='2022-11-01', end_date='2023-02-01', save_path='/data/muscat_data/jaguir26/project1_ucsc_phd/plot_without_nws_ensembles.png', plot_nws_ensembles=False)

plot_ensemble_forecasts_and_data(daily_averages, df_usgs, '2022-12-26', '2023-01-25', daily_avg_nws_forecast_dif_w, weighted_time_series, start_date='2022-11-01', end_date='2023-02-01', save_path='/data/muscat_data/jaguir26/project1_ucsc_phd/plot_without_glofas_ensembles_and_retro.png', plot_glofas_ensembles=False, plot_glofas_retro=False)

plot_ensemble_forecasts_and_data(daily_averages, df_usgs, '2022-12-26', '2023-01-25', daily_avg_nws_forecast_dif_w, weighted_time_series, start_date='2022-11-01', end_date='2023-02-01', save_path='/data/muscat_data/jaguir26/project1_ucsc_phd/plot_without_nws_ensembles_and_retro.png', plot_nws_ensembles=False, plot_nws_retro=False)

plot_ensemble_forecasts_and_data(daily_averages, df_usgs, '2022-12-20', '2023-01-25', daily_avg_nws_forecast_dif_w, weighted_time_series, start_date='2022-11-01', end_date='2023-02-01', save_path='/data/muscat_data/jaguir26/project1_ucsc_phd/plot_all.png')


In [ ]:
import pandas as pd

df_exps_50 = pd.read_csv('/data/muscat_data/jaguir26/project1_ucsc_phd/exps_50.csv')
df_exps_5 = pd.read_csv('/data/muscat_data/jaguir26/project1_ucsc_phd/exps_5.csv')
df_exps_95 = pd.read_csv('/data/muscat_data/jaguir26/project1_ucsc_phd/exps_95.csv')
timestamps = pd.read_csv('/data/muscat_data/jaguir26/project1_ucsc_phd/timestamps.csv')

# Ensure the 'Date' column is in datetime format
timestamps['Date'] = pd.to_datetime(timestamps['Date'])
df_exps_50['Date'] = pd.to_datetime(df_exps_50['Date'])
df_exps_5['Date'] = pd.to_datetime(df_exps_5['Date'])
df_exps_95['Date'] = pd.to_datetime(df_exps_95['Date'])
df_sm_50 = pd.read_csv('/data/muscat_data/jaguir26/project1_ucsc_phd/sm_50.csv')
df_sm_5 = pd.read_csv('/data/muscat_data/jaguir26/project1_ucsc_phd/sm_5.csv')
df_sm_95 = pd.read_csv('/data/muscat_data/jaguir26/project1_ucsc_phd/sm_95.csv')
df_sm_50['Date'] = pd.to_datetime(df_sm_50['Date'])
df_sm_5['Date'] = pd.to_datetime(df_sm_5['Date'])
df_sm_95['Date'] = pd.to_datetime(df_sm_95['Date'])

# Create a new dataframe with required columns
combined_df = pd.DataFrame({
    'Date': timestamps['Date'],
    'Exps_5': df_exps_5['V1'],
    'Exps_50': df_exps_50['V1'],
    'Exps_95': df_exps_95['V1']
})

# Save the combined dataframe to a CSV file
combined_df.to_csv('/data/muscat_data/jaguir26/project1_ucsc_phd/combined_exps.csv', index=False)



In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np

# Load the combined CSV file
combined_df = pd.read_csv('/data/muscat_data/jaguir26/project1_ucsc_phd/combined_exps.csv')
combined_df['Date'] = pd.to_datetime(combined_df['Date'])

# Load the weighted time series for GloFAS
weighted_time_series_path = "/data/muscat_data/jaguir26/project1_ucsc_phd/weighted_time_series.csv"
weighted_time_series = pd.read_csv(weighted_time_series_path)
weighted_time_series['target_date'] = pd.to_datetime(weighted_time_series['target_date'])

def plot_ensemble_forecasts_and_data(daily_averages, df_usgs, start_forecast, end_forecast, daily_avg_nws_forecast_dif_w, weighted_time_series, combined_df, start_date='2018-01-01', end_date=None, save_path=None, plot_glofas_ensembles=True, plot_nws_ensembles=True, plot_glofas_retro=True, plot_nws_retro=True, plot_combined_df=False):
    plt.figure(figsize=(20, 8))
    colormap = plt.cm.cividis
    
    max_ensemble = 10
    colors = {num: colormap(i / max_ensemble) for i, num in enumerate(range(1, max_ensemble + 1))}

    start_date = pd.to_datetime(start_date)
    if end_date:
        end_date = pd.to_datetime(end_date)

    if end_date:
        daily_averages = daily_averages.loc[pd.to_datetime(start_forecast):(pd.to_datetime(start_forecast) + pd.Timedelta(days=10))]
        df_usgs = df_usgs.loc[start_date:end_date]
    else:
        daily_averages = daily_averages.loc[pd.to_datetime(start_forecast):(pd.to_datetime(start_forecast) + pd.Timedelta(days=10))]
        df_usgs = df_usgs.loc[start_date:]

    df_usgs_copy = df_usgs.copy()

    usgs_values = df_usgs_copy['log_discharge_cms']
    df_usgs_copy['standardized'] = usgs_values

    # Split USGS data into before and after start_forecast
    usgs_before = df_usgs_copy.loc[df_usgs_copy.index < start_forecast]
    usgs_after = df_usgs_copy.loc[df_usgs_copy.index >= start_forecast]

    # Plot USGS observations before and after start_forecast with different colors and styles
    plt.plot(usgs_before.index, usgs_before['standardized'], label=f'USGS (Pre {start_forecast})', linestyle='--', color='green', marker='o', markersize=3)
    plt.plot(usgs_after.index, usgs_after['standardized'], label=f'USGS (Post {start_forecast})', linestyle='--', color='lightgreen', marker='o', markersize=6)

    if plot_nws_retro:
        ft = combined_data_cleaned['NWS3.0'].loc[start_date:(pd.to_datetime(start_forecast)- pd.Timedelta(days=1))].copy()
        ft_standardized = ((ft))
        plt.plot(ft.index, ft_standardized, linestyle='solid', color='purple', markersize=1, label='NWS')

    if plot_glofas_retro:
        ft = combined_data_cleaned['GloFAS'].loc[start_date:(pd.to_datetime(start_forecast)- pd.Timedelta(days=1))].copy()
        ft_standardized = ((ft))
        plt.plot(ft.index, ft_standardized, label='GloFAS', linestyle='solid', color='darkorange', markersize=1)
    
    if plot_nws_ensembles:
        ensemble_list = daily_averages.columns
        ensemble_data = daily_averages[ensemble_list]
        mean_ensemble = ensemble_data.mean(axis=1)

        for ensemble in sorted(ensemble_list):
            if ensemble in daily_averages.columns:
                plt.plot(daily_averages.index, daily_averages[ensemble], color='purple', linestyle='-', alpha=0.5, markersize=3)

        plt.plot(daily_averages.index, mean_ensemble, color='darkblue', linewidth=0.3, label='Mean Ensemble')

    if plot_glofas_ensembles:
        # Plot the GloFAS weighted ensembles for the specified date range
        weighted_time_series_filtered = weighted_time_series[(weighted_time_series['target_date'] >= start_forecast) & (weighted_time_series['target_date'] <= (pd.to_datetime(start_forecast) + pd.Timedelta(days=10)))]
        glofas_mean_ensemble = weighted_time_series_filtered.iloc[:, 1:].mean(axis=1)
        for col in weighted_time_series.columns[1:]:
            plt.plot(weighted_time_series_filtered['target_date'], weighted_time_series_filtered[col], linestyle='-', alpha=0.1, color='orange')
        plt.plot(weighted_time_series_filtered['target_date'], glofas_mean_ensemble, color='darkorange', linewidth=0.3, label='GloFAS Mean')
    
    if plot_combined_df:
        # Plot the new time series
        filtered_combined_df = combined_df[(combined_df['Date'] >= start_date) & (combined_df['Date'] < start_forecast)]
        plt.plot(filtered_combined_df['Date'], filtered_combined_df['Exps_5'], color='darkred', label='Quantile 5th', linestyle='--', alpha=0.5)
        plt.plot(filtered_combined_df['Date'], filtered_combined_df['Exps_95'], color='darkblue', label='Quantile 95th', linestyle='--', alpha=0.5)

    # Define cutoff dates and add annotations if within the date range
    cutoff_dates = {
        pd.Timestamp("2018-09-17"): "NWS1.0",
        pd.Timestamp("2019-06-19"): "NWS2.0",
        pd.Timestamp("2021-04-20"): "Hourly data",
        pd.Timestamp("2019-11-25"): "NWS2.1",
        pd.Timestamp("2023-01-10"): "SC22' flood",
        pd.Timestamp("2023-09-20"): "NWS3.0",
        pd.Timestamp(start_forecast): f"Start Forecast - {start_forecast}"
    }
    for date, description in cutoff_dates.items():
        if start_date <= date <= (end_date if end_date else date):
            plt.axvline(date, color='black', linestyle='--', linewidth=1)
            plt.text(date, plt.gca().get_ylim()[1], description, horizontalalignment='center', verticalalignment='bottom', rotation=0, color='black')

    plt.title('')
    plt.xlabel('Date')
    plt.ylabel('Log Flow Discharge')
    plt.legend(title='Legend', loc='upper left')
    plt.gca().xaxis.set_major_locator(mdates.YearLocator())
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.gca().xaxis.set_minor_locator(mdates.MonthLocator())  # Set minor ticks for months
    plt.gca().xaxis.set_minor_formatter(mdates.DateFormatter('%b'))  # Set minor ticks to show month names
    plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=14))  # Set major ticks for specific days
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))  # Format major ticks to show day, month, and year
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300)
    
    plt.show()  


In [ ]:
# Save the plots
plot_ensemble_forecasts_and_data(daily_averages, df_usgs, '2022-12-26', '2023-01-25', daily_avg_nws_forecast_dif_w, weighted_time_series, combined_df, start_date='2022-12-05', end_date='2023-02-01', save_path='/data/muscat_data/jaguir26/project1_ucsc_phd/plot_with_ensembles.png')

plot_ensemble_forecasts_and_data(daily_averages, df_usgs, '2022-12-26', '2023-01-25', daily_avg_nws_forecast_dif_w, weighted_time_series, combined_df, start_date='2022-12-05', end_date='2023-02-01', save_path='/data/muscat_data/jaguir26/project1_ucsc_phd/plot_without_ensembles.png', plot_glofas_ensembles=False, plot_nws_ensembles=False)

plot_ensemble_forecasts_and_data(daily_averages, df_usgs, '2022-12-26', '2023-01-25', daily_avg_nws_forecast_dif_w, weighted_time_series, combined_df, start_date='2022-12-05', end_date='2023-02-01', save_path='/data/muscat_data/jaguir26/project1_ucsc_phd/plot_without_glofas_ensembles.png', plot_glofas_ensembles=False)

plot_ensemble_forecasts_and_data(daily_averages, df_usgs, '2022-12-26', '2023-01-25', daily_avg_nws_forecast_dif_w, weighted_time_series, combined_df, start_date='2022-12-05', end_date='2023-02-01', save_path='/data/muscat_data/jaguir26/project1_ucsc_phd/plot_without_nws_ensembles.png', plot_nws_ensembles=False)

plot_ensemble_forecasts_and_data(daily_averages, df_usgs, '2022-12-26', '2023-01-25', daily_avg_nws_forecast_dif_w, weighted_time_series, combined_df, start_date='2022-12-05', end_date='2023-02-01', save_path='/data/muscat_data/jaguir26/project1_ucsc_phd/plot_without_glofas_ensembles_and_retro.png', plot_glofas_ensembles=False, plot_glofas_retro=False)

plot_ensemble_forecasts_and_data(daily_averages, df_usgs, '2022-12-26', '2023-01-25', daily_avg_nws_forecast_dif_w, weighted_time_series, combined_df, start_date='2022-12-05', end_date='2023-02-01', save_path='/data/muscat_data/jaguir26/project1_ucsc_phd/plot_without_nws_ensembles_and_retro.png', plot_nws_ensembles=False, plot_nws_retro=False)

plot_ensemble_forecasts_and_data(daily_averages, df_usgs, '2022-12-20', '2023-01-25', daily_avg_nws_forecast_dif_w, weighted_time_series, combined_df, start_date='2022-12-05', end_date='2023-02-01', save_path='/data/muscat_data/jaguir26/project1_ucsc_phd/plot_all.png')

plot_ensemble_forecasts_and_data(daily_averages, df_usgs, '2022-12-26', '2023-01-25', daily_avg_nws_forecast_dif_w, weighted_time_series, combined_df, start_date='2022-12-05', end_date='2023-02-01', save_path='/data/muscat_data/jaguir26/project1_ucsc_phd/plot_all_all.png', plot_combined_df=True)


In [ ]:
plot_ensemble_forecasts_and_data(daily_averages, df_usgs, '2022-12-26', '2023-01-25', 
                                 daily_avg_nws_forecast_dif_w, weighted_time_series, combined_df, 
                                 start_date='2022-11-01', end_date='2023-02-01', 
                                 save_path='/data/muscat_data/jaguir26/project1_ucsc_phd/plot_FS1.png', 
                                 plot_glofas_retro=False, plot_nws_retro=False, plot_combined_df=True)

In [ ]:
plot_ensemble_forecasts_and_data(daily_averages, df_usgs, '2022-12-26', '2023-01-25', 
                                 daily_avg_nws_forecast_dif_w, weighted_time_series, combined_df, 
                                 start_date='2022-11-01', end_date='2023-02-01', 
                                 save_path='/data/muscat_data/jaguir26/project1_ucsc_phd/plot_FS2.png', plot_combined_df=True)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np

# Load the combined CSV file
combined_df = pd.read_csv('/data/muscat_data/jaguir26/project1_ucsc_phd/combined_exps.csv')
combined_df['Date'] = pd.to_datetime(combined_df['Date'])

# Load the weighted time series for GloFAS
weighted_time_series_path = "/data/muscat_data/jaguir26/project1_ucsc_phd/weighted_time_series.csv"
weighted_time_series = pd.read_csv(weighted_time_series_path)
weighted_time_series['target_date'] = pd.to_datetime(weighted_time_series['target_date'])

def plot_ensemble_forecasts_and_data(daily_averages, df_usgs, start_forecast, end_forecast, daily_avg_nws_forecast_dif_w, weighted_time_series, combined_df, start_date='2018-01-01', end_date=None, save_path=None, plot_glofas_ensembles=True, plot_nws_ensembles=True, plot_glofas_retro=True, plot_nws_retro=True, plot_combined_df=False):
    plt.figure(figsize=(15, 6))
    colormap = plt.cm.cividis
    
    max_ensemble = 10
    colors = {num: colormap(i / max_ensemble) for i, num in enumerate(range(1, max_ensemble + 1))}

    start_date = pd.to_datetime(start_date)
    if end_date:
        end_date = pd.to_datetime(end_date)

    if end_date:
        daily_averages = daily_averages.loc[pd.to_datetime(start_forecast):(pd.to_datetime(start_forecast) + pd.Timedelta(days=10))]
        df_usgs = df_usgs.loc[start_date:end_date]
        df_usgs = np.log(df_usgs)
        daily_averages = np.log(daily_averages)

    else:
        daily_averages = daily_averages.loc[pd.to_datetime(start_forecast):(pd.to_datetime(start_forecast) + pd.Timedelta(days=10))]
        df_usgs = df_usgs.loc[start_date:]
        df_usgs = np.log(df_usgs)
        daily_averages = np.log(daily_averages)

    df_usgs_copy = df_usgs.copy()

    usgs_values = df_usgs_copy['log_discharge_cms']
    df_usgs_copy['standardized'] = usgs_values

    # Split USGS data into before and after start_forecast
    usgs_before = df_usgs_copy.loc[df_usgs_copy.index < start_forecast]
    usgs_after = df_usgs_copy.loc[df_usgs_copy.index >= start_forecast]

    # Plot USGS observations before and after start_forecast with different colors and styles
    plt.plot(usgs_before.index, usgs_before['standardized'], label=f'USGS (Pre {start_forecast})', linestyle='--', color='green', marker='o', markersize=3)
    plt.plot(usgs_after.index, usgs_after['standardized'], label=f'USGS (Post {start_forecast})', linestyle='--', color='lightgreen', marker='o', markersize=6)

    if plot_nws_retro:
        ft_nws = combined_data_cleaned['NWS3.0'].loc[start_date:(pd.to_datetime(start_forecast)- pd.Timedelta(days=1))].copy()
        ft_standardized = ((ft_nws))
        ft_standardized_nws = np.log(ft_standardized)
        plt.plot(ft_nws.index, ft_standardized_nws, linestyle='solid', color='purple', markersize=1, label='NWS')

    if plot_glofas_retro:
        ft_glofas = combined_data_cleaned['GloFAS'].loc[start_date:(pd.to_datetime(start_forecast)- pd.Timedelta(days=1))].copy()
        ft_standardized = ((ft_glofas))
        ft_standardized_gf = np.log(ft_standardized)
        plt.plot(ft_glofas.index, ft_standardized_gf, label='GloFAS', linestyle='solid', color='darkorange', markersize=1)
    
    if plot_nws_ensembles:
        ensemble_list = daily_averages.columns
        ensemble_data = daily_averages[ensemble_list]
        mean_ensemble_nws = ensemble_data.mean(axis=1)

        for ensemble in sorted(ensemble_list):
            if ensemble in daily_averages.columns:
                plt.plot(daily_averages.index, daily_averages[ensemble] - mean_ensemble_nws.iloc[0] + ft_standardized_nws.iloc[-1], color='purple', linestyle='-', alpha=0.4, markersize=3, linewidth=0.9)
        plt.plot(daily_averages.index, mean_ensemble_nws - mean_ensemble_nws.iloc[0] + ft_standardized_nws.iloc[-1], color='black', linewidth=0.5)

    if plot_glofas_ensembles:
        # Plot the GloFAS weighted ensembles for the specified date range
        weighted_time_series_filtered = weighted_time_series[(weighted_time_series['target_date'] >= start_forecast) & (weighted_time_series['target_date'] <= (pd.to_datetime(start_forecast) + pd.Timedelta(days=28)))]
        glofas_mean_ensemble = weighted_time_series_filtered.iloc[:, 1:].mean(axis=1)

        for col in weighted_time_series.columns[1:]:
            plt.plot(weighted_time_series_filtered['target_date'], weighted_time_series_filtered[col] - glofas_mean_ensemble.iloc[0] + ft_standardized_gf.iloc[-1], linestyle='-', alpha=0.15, color='orange', linewidth=0.9)
        plt.plot(weighted_time_series_filtered['target_date'], glofas_mean_ensemble - glofas_mean_ensemble.iloc[0] + ft_standardized_gf.iloc[-1], color='darkorange', linewidth=1)
    
    if plot_combined_df:
        # Plot the new time series
        filtered_combined_df = combined_df[(combined_df['Date'] >= start_date) & (combined_df['Date'] < start_forecast)]
        plt.plot(filtered_combined_df['Date'], filtered_combined_df['Exps_5'], color='darkred', label='Quantile 5th', linestyle='--', alpha=0.5)
        plt.plot(filtered_combined_df['Date'], filtered_combined_df['Exps_95'], color='darkblue', label='Quantile 95th', linestyle='--', alpha=0.5)

    # Define cutoff dates and add annotations if within the date range
    cutoff_dates = {
        pd.Timestamp("2018-09-17"): "NWS1.0",
        pd.Timestamp("2019-06-19"): "NWS2.0",
        pd.Timestamp("2021-04-20"): "Hourly data",
        pd.Timestamp("2019-11-25"): "NWS2.1",
        pd.Timestamp("2023-01-09"): "SC22' flood",
        pd.Timestamp("2023-09-20"): "NWS3.0",
        pd.Timestamp(start_forecast)- pd.Timedelta(days=1): f"Forecast Starting date - {pd.Timestamp(start_forecast)- pd.Timedelta(days=1)}"
    }
    for date, description in cutoff_dates.items():
        if start_date <= date <= (end_date if end_date else date):
            plt.axvline(date, color='black', linestyle='--', linewidth=1)
            plt.text(date, plt.gca().get_ylim()[1], description, horizontalalignment='center', verticalalignment='bottom', rotation=0, color='black')

    plt.title('')
    plt.xlabel(' ')
    plt.ylabel('Log-og Flow Discharge')
    plt.legend(title='Legend', loc='upper left')
    plt.gca().xaxis.set_major_locator(mdates.YearLocator())
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.gca().xaxis.set_minor_locator(mdates.MonthLocator())  # Set minor ticks for months
    plt.gca().xaxis.set_minor_formatter(mdates.DateFormatter('%b'))  # Set minor ticks to show month names
    plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=5))  # Set major ticks for specific days
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))  # Format major ticks to show day, month, and year
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=600)
    
    plt.show()  


In [ ]:
plot_ensemble_forecasts_and_data(daily_averages, df_usgs, '2022-12-26', '2023-01-25', 
                                 daily_avg_nws_forecast_dif_w, weighted_time_series, combined_df, 
                                 start_date='2022-12-01', end_date='2023-01-24',
                                 save_path='/data/muscat_data/jaguir26/project1_ucsc_phd/plot_FS3.png')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np

# Load the combined CSV file
combined_df = pd.read_csv('/data/muscat_data/jaguir26/project1_ucsc_phd/combined_exps.csv')
combined_df['Date'] = pd.to_datetime(combined_df['Date'])

# Load the weighted time series for GloFAS
weighted_time_series_path = "/data/muscat_data/jaguir26/project1_ucsc_phd/weighted_time_series.csv"
weighted_time_series = pd.read_csv(weighted_time_series_path)
weighted_time_series['target_date'] = pd.to_datetime(weighted_time_series['target_date'])

def plot_ensemble_forecasts_and_data(daily_averages, df_usgs, start_forecast, end_forecast, daily_avg_nws_forecast_dif_w, weighted_time_series, combined_df, start_date='2018-01-01', end_date=None, save_path=None, plot_glofas_ensembles=True, plot_nws_ensembles=True, plot_glofas_retro=True, plot_nws_retro=True, plot_combined_df=False):
    plt.figure(figsize=(20, 8))
    colormap = plt.cm.cividis
    
    max_ensemble = 10
    colors = {num: colormap(i / max_ensemble) for i, num in enumerate(range(1, max_ensemble + 1))}

    start_date = pd.to_datetime(start_date)
    if end_date:
        end_date = pd.to_datetime(end_date)

    if end_date:
        daily_averages = daily_averages.loc[pd.to_datetime(start_forecast):(pd.to_datetime(start_forecast) + pd.Timedelta(days=10))]
        df_usgs = df_usgs.loc[start_date:end_date]
    else:
        daily_averages = daily_averages.loc[pd.to_datetime(start_forecast):(pd.to_datetime(start_forecast) + pd.Timedelta(days=10))]
        df_usgs = df_usgs.loc[start_date:]

    df_usgs_copy = df_usgs.copy()

    usgs_values = df_usgs_copy['log_discharge_cms']
    df_usgs_copy['standardized'] = usgs_values

    # Split USGS data into before and after start_forecast
    usgs_before = df_usgs_copy.loc[df_usgs_copy.index < start_forecast]
    usgs_after = df_usgs_copy.loc[df_usgs_copy.index >= start_forecast]

    # Plot USGS observations before and after start_forecast with different colors and styles
    plt.plot(usgs_before.index, usgs_before['standardized'], label=f'USGS (Pre {start_forecast})', linestyle='--', color='green', marker='o', markersize=3)
    plt.plot(usgs_after.index, usgs_after['standardized'], label=f'USGS (Post {start_forecast})', linestyle='--', color='lightgreen', marker='o', markersize=6)

    if plot_nws_retro:
        ft_nws = combined_data_cleaned['NWS3.0'].loc[start_date:(pd.to_datetime(start_forecast)- pd.Timedelta(days=1))].copy()
        ft_standardized = ((ft_nws))
        plt.plot(ft_nws.index, ft_standardized, linestyle='solid', color='purple', markersize=1, label='NWS')

    if plot_glofas_retro:
        ft_glofas = combined_data_cleaned['GloFAS'].loc[start_date:(pd.to_datetime(start_forecast)- pd.Timedelta(days=1))].copy()
        ft_standardized = ((ft_glofas))
        plt.plot(ft_glofas.index, ft_standardized, label='GloFAS', linestyle='solid', color='darkorange', markersize=1)
    
    if plot_nws_ensembles:
        ensemble_list = daily_averages.columns
        ensemble_data = daily_averages[ensemble_list]
        mean_ensemble_nws = ensemble_data.mean(axis=1)

        for ensemble in sorted(ensemble_list):
            if ensemble in daily_averages.columns:
                plt.plot(daily_averages.index, daily_averages[ensemble] - mean_ensemble_nws.iloc[0] + usgs_before['standardized'].iloc[-1], color='purple', linestyle='-', alpha=0.5, markersize=3)

        plt.plot(daily_averages.index, mean_ensemble_nws - mean_ensemble_nws.iloc[0] + usgs_before['standardized'].iloc[-1], color='darkblue', linewidth=0.3, label='Mean Ensemble (Adjusted)')

    if plot_glofas_ensembles:
        # Plot the GloFAS weighted ensembles for the specified date range
        weighted_time_series_filtered = weighted_time_series[(weighted_time_series['target_date'] >= start_forecast) & (weighted_time_series['target_date'] <= (pd.to_datetime(start_forecast) + pd.Timedelta(days=30)))]
        glofas_mean_ensemble = weighted_time_series_filtered.iloc[:, 1:].mean(axis=1)

        for col in weighted_time_series.columns[1:]:
            plt.plot(weighted_time_series_filtered['target_date'], weighted_time_series_filtered[col] - glofas_mean_ensemble.iloc[0] + usgs_before['standardized'].iloc[-1], linestyle='-', alpha=0.1, color='orange')
        
        plt.plot(weighted_time_series_filtered['target_date'], glofas_mean_ensemble - glofas_mean_ensemble.iloc[0] + usgs_before['standardized'].iloc[-1], color='darkorange', linewidth=0.3, label='GloFAS Mean (Adjusted)')
    
    if plot_combined_df:
        # Plot the new time series
        filtered_combined_df = combined_df[(combined_df['Date'] >= start_date) & (combined_df['Date'] < start_forecast)]
        plt.plot(filtered_combined_df['Date'], filtered_combined_df['Exps_5'], color='darkred', label='Quantile 5th', linestyle='--', alpha=0.5)
        plt.plot(filtered_combined_df['Date'], filtered_combined_df['Exps_95'], color='darkblue', label='Quantile 95th', linestyle='--', alpha=0.5)

    # Define cutoff dates and add annotations if within the date range
    cutoff_dates = {
        pd.Timestamp("2018-09-17"): "NWS1.0",
        pd.Timestamp("2019-06-19"): "NWS2.0",
        pd.Timestamp("2021-04-20"): "Hourly data",
        pd.Timestamp("2019-11-25"): "NWS2.1",
        pd.Timestamp("2023-01-10"): "SC22' flood",
        pd.Timestamp("2023-09-20"): "NWS3.0",
        pd.Timestamp(start_forecast): f"Start Forecast - {start_forecast}"
    }
    for date, description in cutoff_dates.items():
        if start_date <= date <= (end_date if end_date else date):
            plt.axvline(date, color='black', linestyle='--', linewidth=1)
            plt.text(date, plt.gca().get_ylim()[1], description, horizontalalignment='center', verticalalignment='bottom', rotation=0, color='black')

    plt.title('')
    plt.xlabel('Date')
    plt.ylabel('Log Flow Discharge')
    plt.legend(title='Legend', loc='upper left')
    plt.gca().xaxis.set_major_locator(mdates.YearLocator())
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.gca().xaxis.set_minor_locator(mdates.MonthLocator())  # Set minor ticks for months
    plt.gca().xaxis.set_minor_formatter(mdates.DateFormatter('%b'))  # Set minor ticks to show month names
    plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=14))  # Set major ticks for specific days
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))  # Format major ticks to show day, month, and year
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300)
    
    plt.show()  


In [ ]:
plot_ensemble_forecasts_and_data(daily_averages, df_usgs, '2022-12-26', '2023-01-25', 
                                 daily_avg_nws_forecast_dif_w, weighted_time_series, combined_df, 
                                 start_date='2022-11-01', end_date='2023-02-01', 
                                 plot_combined_df=True, plot_glofas_retro=False, plot_nws_retro=False,
                                 save_path='/data/muscat_data/jaguir26/project1_ucsc_phd/plot_FS4.png')


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np

# Load the combined CSV file
combined_df = pd.read_csv('/data/muscat_data/jaguir26/project1_ucsc_phd/combined_exps.csv')
combined_df['Date'] = pd.to_datetime(combined_df['Date'])

# Load the weighted time series for GloFAS
weighted_time_series_path = "/data/muscat_data/jaguir26/project1_ucsc_phd/weighted_time_series.csv"
weighted_time_series = pd.read_csv(weighted_time_series_path)
weighted_time_series['target_date'] = pd.to_datetime(weighted_time_series['target_date'])

def plot_ensemble_forecasts_and_data(daily_averages, df_usgs, start_forecast, end_forecast, daily_avg_nws_forecast_dif_w, weighted_time_series, combined_df, start_date='2018-01-01', end_date=None, save_path=None, plot_glofas_ensembles=True, plot_nws_ensembles=True, plot_glofas_retro=True, plot_nws_retro=True, plot_combined_df=False):
    plt.figure(figsize=(20, 8))
    colormap = plt.cm.cividis
    
    max_ensemble = 10
    colors = {num: colormap(i / max_ensemble) for i, num in enumerate(range(1, max_ensemble + 1))}

    start_date = pd.to_datetime(start_date)
    if end_date:
        end_date = pd.to_datetime(end_date)

    if end_date:
        daily_averages = daily_averages.loc[pd.to_datetime(start_forecast):(pd.to_datetime(start_forecast) + pd.Timedelta(days=10))]
        df_usgs = df_usgs.loc[start_date:end_date]
    else:
        daily_averages = daily_averages.loc[pd.to_datetime(start_forecast):(pd.to_datetime(start_forecast) + pd.Timedelta(days=10))]
        df_usgs = df_usgs.loc[start_date:]

    df_usgs_copy = df_usgs.copy()

    usgs_values = df_usgs_copy['log_discharge_cms']
    df_usgs_copy['standardized'] = usgs_values

    # Split USGS data into before and after start_forecast
    usgs_before = df_usgs_copy.loc[df_usgs_copy.index < start_forecast]
    usgs_after = df_usgs_copy.loc[df_usgs_copy.index >= start_forecast]

    # Plot USGS observations before and after start_forecast with different colors and styles
    plt.plot(usgs_before.index, usgs_before['standardized'], label=f'USGS (Pre {start_forecast})', linestyle='--', color='green', marker='o', markersize=3)
    plt.plot(usgs_after.index, usgs_after['standardized'], label=f'USGS (Post {start_forecast})', linestyle='--', color='lightgreen', marker='o', markersize=6)

    if plot_nws_retro:
        filtered_combined_df = combined_df[(combined_df['Date'] >= start_date) & (combined_df['Date'] < start_forecast)]
        ft_standardized = ((ft_nws))
        plt.plot(ft_nws.index, ft_standardized, linestyle='solid', color='purple', markersize=1, label='NWS')

    if plot_glofas_retro:
        ft_glofas = combined_data_cleaned['GloFAS'].loc[start_date:(pd.to_datetime(start_forecast)- pd.Timedelta(days=1))].copy()
        ft_standardized = ((ft_glofas))
        plt.plot(ft_glofas.index, ft_standardized, label='GloFAS', linestyle='solid', color='darkorange', markersize=1)
    
    if plot_combined_df:
        # Plot the new time series
        filtered_combined_df = combined_df[(combined_df['Date'] >= start_date) & (combined_df['Date'] < start_forecast)]
        plt.plot(filtered_combined_df['Date'], filtered_combined_df['Exps_5'], color='darkred', label='Quantile 5th', linestyle='--', alpha=0.5)
        plt.plot(filtered_combined_df['Date'], filtered_combined_df['Exps_95'], color='darkblue', label='Quantile 95th', linestyle='--', alpha=0.5)


    if plot_nws_ensembles:
        ensemble_list = daily_averages.columns
        ensemble_data = daily_averages[ensemble_list]
        mean_ensemble_nws = ensemble_data.mean(axis=1)

        for ensemble in sorted(ensemble_list):
            if ensemble in daily_averages.columns:
                plt.plot(daily_averages.index, daily_averages[ensemble] - mean_ensemble_nws.iloc[0] + filtered_combined_df['Exps_95'].iloc[-1], color='darkblue', linestyle='-', alpha=0.5, markersize=3)

        plt.plot(daily_averages.index, mean_ensemble_nws - mean_ensemble_nws.iloc[0] + filtered_combined_df['Exps_95'].iloc[-1], color='darkblue', linewidth=0.3, label='Mean Ensemble (Adjusted)')

    if plot_glofas_ensembles:
        # Plot the GloFAS weighted ensembles for the specified date range
        weighted_time_series_filtered = weighted_time_series[(weighted_time_series['target_date'] >= start_forecast) & (weighted_time_series['target_date'] <= (pd.to_datetime(start_forecast) + pd.Timedelta(days=30)))]
        glofas_mean_ensemble = weighted_time_series_filtered.iloc[:, 1:].mean(axis=1)

        for col in weighted_time_series.columns[1:]:
            plt.plot(weighted_time_series_filtered['target_date'], weighted_time_series_filtered[col] - glofas_mean_ensemble.iloc[0] + filtered_combined_df['Exps_95'].iloc[-1], linestyle='-', alpha=0.1, color='darkblue')
        
        plt.plot(weighted_time_series_filtered['target_date'], glofas_mean_ensemble - glofas_mean_ensemble.iloc[0] + filtered_combined_df['Exps_95'].iloc[-1], color='darkblue', linewidth=0.3, label='GloFAS Mean (Adjusted)')
    
    
    if plot_nws_ensembles:
        ensemble_list = daily_averages.columns
        ensemble_data = daily_averages[ensemble_list]
        mean_ensemble_nws = ensemble_data.mean(axis=1)

        for ensemble in sorted(ensemble_list):
            if ensemble in daily_averages.columns:
                plt.plot(daily_averages.index, daily_averages[ensemble] - mean_ensemble_nws.iloc[0] + filtered_combined_df['Exps_5'].iloc[-1], color='darkred', linestyle='-', alpha=0.5, markersize=3)

        plt.plot(daily_averages.index, mean_ensemble_nws - mean_ensemble_nws.iloc[0] + filtered_combined_df['Exps_5'].iloc[-1], color='darkred', linewidth=0.3, label='Mean Ensemble (Adjusted)')

    if plot_glofas_ensembles:
        # Plot the GloFAS weighted ensembles for the specified date range
        weighted_time_series_filtered = weighted_time_series[(weighted_time_series['target_date'] >= start_forecast) & (weighted_time_series['target_date'] <= (pd.to_datetime(start_forecast) + pd.Timedelta(days=30)))]
        glofas_mean_ensemble = weighted_time_series_filtered.iloc[:, 1:].mean(axis=1)

        for col in weighted_time_series.columns[1:]:
            plt.plot(weighted_time_series_filtered['target_date'], weighted_time_series_filtered[col] - glofas_mean_ensemble.iloc[0] + filtered_combined_df['Exps_5'].iloc[-1], linestyle='-', alpha=0.1, color='darkred')
        
        plt.plot(weighted_time_series_filtered['target_date'], glofas_mean_ensemble - glofas_mean_ensemble.iloc[0] + filtered_combined_df['Exps_5'].iloc[-1], color='darkred', linewidth=0.3, label='GloFAS Mean (Adjusted)')
    
    # Define cutoff dates and add annotations if within the date range
    cutoff_dates = {
        pd.Timestamp("2018-09-17"): "NWS1.0",
        pd.Timestamp("2019-06-19"): "NWS2.0",
        pd.Timestamp("2021-04-20"): "Hourly data",
        pd.Timestamp("2019-11-25"): "NWS2.1",
        pd.Timestamp("2023-01-10"): "SC22' flood",
        pd.Timestamp("2023-09-20"): "NWS3.0",
        pd.Timestamp(start_forecast): f"Start Forecast - {start_forecast}"
    }
    for date, description in cutoff_dates.items():
        if start_date <= date <= (end_date if end_date else date):
            plt.axvline(date, color='black', linestyle='--', linewidth=1)
            plt.text(date, plt.gca().get_ylim()[1], description, horizontalalignment='center', verticalalignment='bottom', rotation=0, color='black')

    plt.title('')
    plt.xlabel('Date')
    plt.ylabel('Log Flow Discharge')
    plt.legend(title='Legend', loc='upper left')
    plt.gca().xaxis.set_major_locator(mdates.YearLocator())
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.gca().xaxis.set_minor_locator(mdates.MonthLocator())  # Set minor ticks for months
    plt.gca().xaxis.set_minor_formatter(mdates.DateFormatter('%b'))  # Set minor ticks to show month names
    plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=14))  # Set major ticks for specific days
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))  # Format major ticks to show day, month, and year
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300)
    
    plt.show()  


In [ ]:
plot_ensemble_forecasts_and_data(daily_averages, df_usgs, '2022-12-26', '2023-01-25', 
                                 daily_avg_nws_forecast_dif_w, weighted_time_series, combined_df, 
                                 start_date='2022-11-01', end_date='2023-02-01', 
                                 plot_combined_df=True, plot_glofas_retro=False, plot_nws_retro=False,
                                 save_path='/data/muscat_data/jaguir26/project1_ucsc_phd/plot_FS5.png')


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np

# Load the combined CSV file
combined_df = pd.read_csv('/data/muscat_data/jaguir26/project1_ucsc_phd/combined_exps.csv')
combined_df['Date'] = pd.to_datetime(combined_df['Date'])

# Load the weighted time series for GloFAS
weighted_time_series_path = "/data/muscat_data/jaguir26/project1_ucsc_phd/weighted_time_series.csv"
weighted_time_series = pd.read_csv(weighted_time_series_path)
weighted_time_series['target_date'] = pd.to_datetime(weighted_time_series['target_date'])

def plot_ensemble_forecasts_and_data(daily_averages, df_usgs, start_forecast, end_forecast, daily_avg_nws_forecast_dif_w, weighted_time_series, combined_df, start_date='2018-01-01', end_date=None, save_path=None, plot_glofas_ensembles=True, plot_nws_ensembles=True, plot_glofas_retro=True, plot_nws_retro=True, plot_combined_df=False):
    plt.figure(figsize=(20, 8))
    colormap = plt.cm.cividis
    
    max_ensemble = 10
    colors = {num: colormap(i / max_ensemble) for i, num in enumerate(range(1, max_ensemble + 1))}

    start_date = pd.to_datetime(start_date)
    if end_date:
        end_date = pd.to_datetime(end_date)

    if end_date:
        daily_averages = daily_averages.loc[pd.to_datetime(start_forecast):(pd.to_datetime(start_forecast) + pd.Timedelta(days=10))]
        df_usgs = df_usgs.loc[start_date:end_date]
    else:
        daily_averages = daily_averages.loc[pd.to_datetime(start_forecast):(pd.to_datetime(start_forecast) + pd.Timedelta(days=10))]
        df_usgs = df_usgs.loc[start_date:]

    df_usgs_copy = df_usgs.copy()

    usgs_values = df_usgs_copy['log_discharge_cms']
    df_usgs_copy['standardized'] = usgs_values

    # Split USGS data into before and after start_forecast
    usgs_before = df_usgs_copy.loc[df_usgs_copy.index < start_forecast]
    usgs_after = df_usgs_copy.loc[df_usgs_copy.index >= start_forecast]

    # Plot USGS observations before and after start_forecast with different colors and styles
    plt.plot(usgs_before.index, usgs_before['standardized'], label=f'USGS (Pre {start_forecast})', linestyle='--', color='green', marker='o', markersize=3)
    plt.plot(usgs_after.index, usgs_after['standardized'], label=f'USGS (Post {start_forecast})', linestyle='--', color='lightgreen', marker='o', markersize=6)

    if plot_nws_retro:
        ft_nws = combined_data_cleaned['NWS3.0'].loc[start_date:(pd.to_datetime(start_forecast)- pd.Timedelta(days=1))].copy()
        ft_standardized = ((ft_nws))
        plt.plot(ft_nws.index, ft_standardized, linestyle='solid', color='purple', markersize=1, label='NWS')

    if plot_glofas_retro:
        ft_glofas = combined_data_cleaned['GloFAS'].loc[start_date:(pd.to_datetime(start_forecast)- pd.Timedelta(days=1))].copy()
        ft_standardized = ((ft_glofas))
        plt.plot(ft_glofas.index, ft_standardized, label='GloFAS', linestyle='solid', color='darkorange', markersize=1)
    
    if plot_combined_df:
        # Plot the new time series
        filtered_combined_df = combined_df[(combined_df['Date'] >= start_date) & (combined_df['Date'] < start_forecast)]
        plt.plot(filtered_combined_df['Date'], filtered_combined_df['Exps_5'], color='darkred', label='Quantile 5th', linestyle='--', alpha=0.5)
        plt.plot(filtered_combined_df['Date'], filtered_combined_df['Exps_95'], color='darkblue', label='Quantile 95th', linestyle='--', alpha=0.5)


    if plot_nws_ensembles:
        ensemble_list = daily_averages.columns
        ensemble_data = daily_averages[ensemble_list]
        mean_ensemble_nws = ensemble_data.mean(axis=1)

        # for ensemble in sorted(ensemble_list):
        #     if ensemble in daily_averages.columns:
        #         plt.plot(daily_averages.index, daily_averages[ensemble] - mean_ensemble_nws.iloc[0] + filtered_combined_df['Exps_95'].iloc[-1], color='darkblue', linestyle='-', alpha=0.5, markersize=3)

        plt.plot(daily_averages.index, mean_ensemble_nws - mean_ensemble_nws.iloc[0] + filtered_combined_df['Exps_95'].iloc[-1], color='darkblue', linewidth=1, label='Mean Ensemble (Adjusted)')

    if plot_glofas_ensembles:
        # Plot the GloFAS weighted ensembles for the specified date range
        weighted_time_series_filtered = weighted_time_series[(weighted_time_series['target_date'] >= start_forecast) & (weighted_time_series['target_date'] <= (pd.to_datetime(start_forecast) + pd.Timedelta(days=10)))]
        glofas_mean_ensemble = weighted_time_series_filtered.iloc[:, 1:].mean(axis=1)

        # for col in weighted_time_series.columns[1:]:
        #     plt.plot(weighted_time_series_filtered['target_date'], weighted_time_series_filtered[col] - glofas_mean_ensemble.iloc[0] + filtered_combined_df['Exps_95'].iloc[-1], linestyle='-', alpha=0.1, color='darkblue')
        
        plt.plot(weighted_time_series_filtered['target_date'], glofas_mean_ensemble - glofas_mean_ensemble.iloc[0] + filtered_combined_df['Exps_95'].iloc[-1], color='darkblue', linewidth=1, label='GloFAS Mean (Adjusted)')
    
    
    if plot_nws_ensembles:
        ensemble_list = daily_averages.columns
        ensemble_data = daily_averages[ensemble_list]
        mean_ensemble_nws = ensemble_data.mean(axis=1)

        # for ensemble in sorted(ensemble_list):
        #     if ensemble in daily_averages.columns:
        #         plt.plot(daily_averages.index, daily_averages[ensemble] - mean_ensemble_nws.iloc[0] + filtered_combined_df['Exps_5'].iloc[-1], color='darkred', linestyle='-', alpha=0.5, markersize=3)

        plt.plot(daily_averages.index, mean_ensemble_nws - mean_ensemble_nws.iloc[0] + filtered_combined_df['Exps_5'].iloc[-1], color='darkred', linewidth=1, label='Mean Ensemble (Adjusted)')

    if plot_glofas_ensembles:
        # Plot the GloFAS weighted ensembles for the specified date range
        weighted_time_series_filtered = weighted_time_series[(weighted_time_series['target_date'] >= start_forecast) & (weighted_time_series['target_date'] <= (pd.to_datetime(start_forecast) + pd.Timedelta(days=10)))]
        glofas_mean_ensemble = weighted_time_series_filtered.iloc[:, 1:].mean(axis=1)

        # for col in weighted_time_series.columns[1:]:
        #     plt.plot(weighted_time_series_filtered['target_date'], weighted_time_series_filtered[col] - glofas_mean_ensemble.iloc[0] + filtered_combined_df['Exps_5'].iloc[-1], linestyle='-', alpha=0.1, color='darkred')
        
        plt.plot(weighted_time_series_filtered['target_date'], glofas_mean_ensemble - glofas_mean_ensemble.iloc[0] + filtered_combined_df['Exps_5'].iloc[-1], color='darkred', linewidth=1, label='GloFAS Mean (Adjusted)')
    
    # Define cutoff dates and add annotations if within the date range
    cutoff_dates = {
        pd.Timestamp("2018-09-17"): "NWS1.0",
        pd.Timestamp("2019-06-19"): "NWS2.0",
        pd.Timestamp("2021-04-20"): "Hourly data",
        pd.Timestamp("2019-11-25"): "NWS2.1",
        pd.Timestamp("2023-01-10"): "SC22' flood",
        pd.Timestamp("2023-09-20"): "NWS3.0",
        pd.Timestamp(start_forecast): f"Start Forecast - {start_forecast}"
    }
    for date, description in cutoff_dates.items():
        if start_date <= date <= (end_date if end_date else date):
            plt.axvline(date, color='black', linestyle='--', linewidth=1)
            plt.text(date, plt.gca().get_ylim()[1], description, horizontalalignment='center', verticalalignment='bottom', rotation=0, color='black')

    plt.title('')
    plt.xlabel('Date')
    plt.ylabel('Log Flow Discharge')
    plt.legend(title='Legend', loc='upper left')
    plt.gca().xaxis.set_major_locator(mdates.YearLocator())
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.gca().xaxis.set_minor_locator(mdates.MonthLocator())  # Set minor ticks for months
    plt.gca().xaxis.set_minor_formatter(mdates.DateFormatter('%b'))  # Set minor ticks to show month names
    plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=14))  # Set major ticks for specific days
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))  # Format major ticks to show day, month, and year
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300)
    
    plt.show()  


In [ ]:
plot_ensemble_forecasts_and_data(daily_averages, df_usgs,'2022-12-26', '2023-01-25', 
                                 daily_avg_nws_forecast_dif_w, weighted_time_series, combined_df, 
                                 start_date='2022-11-01', end_date='2023-02-01', 
                                 plot_combined_df=True, plot_glofas_retro=False, plot_nws_retro=False,
                                 save_path='/data/muscat_data/jaguir26/project1_ucsc_phd/plot_FS6.png')


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np

# Load the combined CSV file
combined_df = pd.read_csv('/data/muscat_data/jaguir26/project1_ucsc_phd/combined_exps.csv')
combined_df['Date'] = pd.to_datetime(combined_df['Date'])

# Load the weighted time series for GloFAS
weighted_time_series_path = "/data/muscat_data/jaguir26/project1_ucsc_phd/weighted_time_series.csv"
weighted_time_series = pd.read_csv(weighted_time_series_path)
weighted_time_series['target_date'] = pd.to_datetime(weighted_time_series['target_date'])

def plot_ensemble_forecasts_and_data(daily_averages, df_usgs, start_forecast, end_forecast, daily_avg_nws_forecast_dif_w, weighted_time_series, combined_df, start_date='2018-01-01', end_date=None, save_path=None, plot_glofas_ensembles=True, plot_nws_ensembles=True, plot_glofas_retro=True, plot_nws_retro=True, plot_combined_df=False):
    plt.figure(figsize=(20, 8))
    colormap = plt.cm.cividis
    
    max_ensemble = 10
    colors = {num: colormap(i / max_ensemble) for i, num in enumerate(range(1, max_ensemble + 1))}

    start_date = pd.to_datetime(start_date)
    if end_date:
        end_date = pd.to_datetime(end_date)

    if end_date:
        daily_averages = daily_averages.loc[pd.to_datetime(start_forecast):(pd.to_datetime(start_forecast) + pd.Timedelta(days=10))]
        df_usgs = df_usgs.loc[start_date:end_date]
    else:
        daily_averages = daily_averages.loc[pd.to_datetime(start_forecast):(pd.to_datetime(start_forecast) + pd.Timedelta(days=10))]
        df_usgs = df_usgs.loc[start_date:]

    df_usgs_copy = df_usgs.copy()

    usgs_values = df_usgs_copy['log_discharge_cms']
    df_usgs_copy['standardized'] = usgs_values

    # Split USGS data into before and after start_forecast
    usgs_before = df_usgs_copy.loc[df_usgs_copy.index < start_forecast]
    usgs_after = df_usgs_copy.loc[df_usgs_copy.index >= start_forecast]

    # Plot USGS observations before and after start_forecast with different colors and styles
    plt.plot(usgs_before.index, usgs_before['standardized'], label=f'USGS (Pre {start_forecast})', linestyle='--', color='green', marker='o', markersize=3)
    plt.plot(usgs_after.index, usgs_after['standardized'], label=f'USGS (Post {start_forecast})', linestyle='--', color='lightgreen', marker='o', markersize=6)

    if plot_nws_retro:
        ft_nws = combined_data_cleaned['NWS3.0'].loc[start_date:(pd.to_datetime(start_forecast)- pd.Timedelta(days=1))].copy()
        ft_standardized = ((ft_nws))
        plt.plot(ft_nws.index, ft_standardized, linestyle='solid', color='purple', markersize=1, label='NWS')

    if plot_glofas_retro:
        ft_glofas = combined_data_cleaned['GloFAS'].loc[start_date:(pd.to_datetime(start_forecast)- pd.Timedelta(days=1))].copy()
        ft_standardized = ((ft_glofas))
        plt.plot(ft_glofas.index, ft_standardized, label='GloFAS', linestyle='solid', color='darkorange', markersize=1)
    
    if plot_combined_df:
        # Plot the new time series
        filtered_combined_df = combined_df[(combined_df['Date'] >= start_date) & (combined_df['Date'] < start_forecast)]
        plt.plot(filtered_combined_df['Date'], filtered_combined_df['Exps_5'], color='darkred', label='Quantile 5th', linestyle='--', alpha=0.5)
        plt.plot(filtered_combined_df['Date'], filtered_combined_df['Exps_95'], color='darkblue', label='Quantile 95th', linestyle='--', alpha=0.5)
        plt.plot(filtered_combined_df['Date'], filtered_combined_df['Exps_50'], color='olivedrab', label='Quantile 50th', linestyle='--', alpha=0.5)

    if plot_nws_ensembles:
        ensemble_list = daily_averages.columns
        ensemble_data = daily_averages[ensemble_list]
        mean_ensemble_nws = ensemble_data.mean(axis=1)
        adjusted_mean_nws_95 = mean_ensemble_nws - mean_ensemble_nws.iloc[0] + filtered_combined_df['Exps_95'].iloc[-1]
        adjusted_mean_nws_5 = mean_ensemble_nws - mean_ensemble_nws.iloc[0] + filtered_combined_df['Exps_5'].iloc[-1]
        adjusted_mean_nws_50 = mean_ensemble_nws - mean_ensemble_nws.iloc[0] + filtered_combined_df['Exps_50'].iloc[-1]

    if plot_glofas_ensembles:
        weighted_time_series_filtered = weighted_time_series[
            (weighted_time_series['target_date'] >= start_forecast) & 
            (weighted_time_series['target_date'] <= (pd.to_datetime(start_forecast) + pd.Timedelta(days=10)))
        ]
        glofas_mean_ensemble = weighted_time_series_filtered.iloc[:, 1:].mean(axis=1)
        glofas_mean_ensemble.index = weighted_time_series_filtered['target_date']
        adjusted_mean_glofas_95 = glofas_mean_ensemble - glofas_mean_ensemble.iloc[0] + filtered_combined_df['Exps_95'].iloc[-1]
        adjusted_mean_glofas_5 = glofas_mean_ensemble - glofas_mean_ensemble.iloc[0] + filtered_combined_df['Exps_5'].iloc[-1]
        adjusted_mean_glofas_50 = glofas_mean_ensemble - glofas_mean_ensemble.iloc[0] + filtered_combined_df['Exps_50'].iloc[-1]

    # Compute and plot the combined means
    common_index = daily_averages.index.intersection(weighted_time_series_filtered['target_date'])
    a1 = adjusted_mean_nws_95.reindex(common_index)
    a2 = adjusted_mean_glofas_95.reindex(common_index)
    a3 = 0.9 * a1 + 0.1 * a2

    b1 = adjusted_mean_nws_5.reindex(common_index)
    b2 = adjusted_mean_glofas_5.reindex(common_index)
    b3 = 0.6 * b1 + 0.4 * b2
    
    c1 = adjusted_mean_nws_50.reindex(common_index)
    c2 = adjusted_mean_glofas_50.reindex(common_index)
    c3 = 0.5 * c1 + 0.5 * c2

    plt.plot(common_index, a3, color='darkblue', linewidth=2, label='Adjusted Quantile Synth - 95th')
    plt.plot(common_index, b3, color='darkred', linewidth=2, label='Adjusted Quantile Synth - 5th')
    plt.plot(common_index, c3, color='olivedrab', linewidth=2, label='Adjusted Quantile Synth - 50th')


    # Add shaded area between the combined means after the cutoff date
    plt.fill_between(common_index, a3, b3, where=(a3 > b3), facecolor='pink', alpha=0.3)
    plt.fill_between(common_index, a3, b3, where=(a3 < b3), facecolor='pink', alpha=0.3)

    # Define cutoff dates and add annotations if within the date range
    cutoff_dates = {
        pd.Timestamp("2018-09-17"): "NWS1.0",
        pd.Timestamp("2019-06-19"): "NWS2.0",
        pd.Timestamp("2021-04-20"): "Hourly data",
        pd.Timestamp("2019-11-25"): "NWS2.1",
        pd.Timestamp("2023-01-10"): "SC22' flood",
        pd.Timestamp("2023-09-20"): "NWS3.0",
        pd.Timestamp(start_forecast): f"Start Forecast - {start_forecast}"
    }
    for date, description in cutoff_dates.items():
        if start_date <= date <= (end_date if end_date else date):
            plt.axvline(date, color='black', linestyle='--', linewidth=1)
            plt.text(date, plt.gca().get_ylim()[1], description, horizontalalignment='center', verticalalignment='bottom', rotation=0, color='black')

    plt.title('')
    plt.xlabel('Date')
    plt.ylabel('Log Flow Discharge')
    plt.legend(title='Legend', loc='upper left')
    plt.gca().xaxis.set_major_locator(mdates.YearLocator())
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.gca().xaxis.set_minor_locator(mdates.MonthLocator())  # Set minor ticks for months
    plt.gca().xaxis.set_minor_formatter(mdates.DateFormatter('%b'))  # Set minor ticks to show month names
    plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=14))  # Set major ticks for specific days
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))  # Format major ticks to show day, month, and year
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300)
    
    plt.show()  


In [ ]:
plot_ensemble_forecasts_and_data(daily_averages, df_usgs, '2022-12-26', '2023-01-25', 
                                 daily_avg_nws_forecast_dif_w, weighted_time_series, combined_df, 
                                 start_date='2022-11-01', end_date='2023-02-01', 
                                 plot_combined_df=True, plot_glofas_retro=False, plot_nws_retro=False,
                                 save_path='/data/muscat_data/jaguir26/project1_ucsc_phd/plot_FS7.png')


In [ ]:
import pandas as pd
from functools import reduce

# Rename columns in the DataFrames
daily_avg_usgs.rename(columns={'Daily_Avg_Log_Streamflow': 'USGS'}, inplace=True)
daily_avg_nws.rename(columns={'Daily_Avg_Log_Streamflow': 'NWS3.0'}, inplace=True)
daily_avg_glofas.rename(columns={'Daily_Avg_Log_Streamflow': 'GloFAS'}, inplace=True)

# Step 1: Combine DataFrames by their dates
def combine_dataframes(df_list, join_type='inner'):
    df_merged = reduce(lambda left, right: pd.merge(left, right, on='Date', how=join_type), df_list)
    return df_merged

# Ensure there are no NAs and retain the largest continuous non-NA segment
def remove_na_and_truncate(df, end_date=None):
    df_non_na = df.dropna()
    if not df_non_na.empty:
        # Find the last date without NA
        last_valid_date = df_non_na['Date'].max()
        # Truncate the DataFrame up to the last valid date
        df_truncated = df[df['Date'] <= last_valid_date]
    else:
        df_truncated = pd.DataFrame()
    # If end_date is specified, further truncate the DataFrame
    if end_date:
        end_date = pd.to_datetime(end_date)
        df_truncated = df_truncated[df_truncated['Date'] <= end_date]
    return df_truncated

# Combine daily_avg_usgs, daily_avg_nws, and daily_avg_glofas
df_list = [daily_avg_usgs, daily_avg_nws, daily_avg_glofas]
combined_df = combine_dataframes(df_list)

# Remove NAs and retain the largest continuous non-NA segment, with an optional end_date
end_date = '2023-06-01'  # Example end date
cleaned_combined_df = remove_na_and_truncate(combined_df, end_date)

# Store the DataFrames with the Date column intact
base_path = '/data/muscat_data/jaguir26/project1_ucsc_phd/'

combined_file_path = f'{base_path}retros_{end_date}.csv'
cleaned_combined_df.to_csv(combined_file_path, index=False)

glofas_ens_file_path = f'{base_path}glofas_ens_{end_date}.csv'
daily_avg_glofas_ens.to_csv(glofas_ens_file_path, index=False)

nws_forecast_file_path = f'{base_path}nws_ens_{end_date}.csv'
daily_avg_nws_forecast.to_csv(nws_forecast_file_path, index=False)

print(f"Combined DataFrame saved to: {combined_file_path}")
print(f"GLOFAS Ensemble DataFrame saved to: {glofas_ens_file_path}")
print(f"NWS Forecast DataFrame saved to: {nws_forecast_file_path}")

# Paths to the CSV files
combined_file_path = f'{base_path}retros_{end_date}.csv'
glofas_ens_file_path = f'{base_path}glofas_ens_{end_date}.csv'
nws_forecast_file_path = f'{base_path}nws_ens_{end_date}.csv'

# Read the CSV files into DataFrames
combined_df = pd.read_csv(combined_file_path)
glofas_ens_df = pd.read_csv(glofas_ens_file_path)
nws_forecast_df = pd.read_csv(nws_forecast_file_path)

# Display the content of the DataFrames
print("Combined DataFrame:")
print(combined_df.head())

print("\nGLOFAS Ensemble DataFrame:")
print(glofas_ens_df.head())

print("\nNWS Forecast DataFrame:")
print(nws_forecast_df.head())


# Climate Indeces

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set up the plotting style
sns.set(style="whitegrid")
plt.rcParams.update({
    "figure.figsize": (20, 15),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "lines.linewidth": 2,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "font.family": "serif"
})

# Custom color palette
custom_colors = sns.color_palette("muted", n_colors=12)
friendly_blue = sns.color_palette("Blues")[5]

# Function to plot each time series in subplots
def plot_time_series_subplots(df, columns_dict, start_date, end_date, file_path, colors):
    # Filter the DataFrame for the date range
    df_filtered = df[(df['Date'] >= start_date) & (df['Date'] <= end_date)]
    
    # Create subplots
    fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(20, 15), sharex=True)
    axes = axes.flatten()
    
    for idx, (column, title) in enumerate(columns_dict.items()):
        sns.lineplot(data=df_filtered, x='Date', y=column, ax=axes[idx], color=colors[idx % len(colors)])
        axes[idx].set_title(title)
        axes[idx].set_xlabel('Date')
        axes[idx].set_ylabel(column)
        axes[idx].tick_params(axis='x', rotation=45)
    
    # Adjust layout
    plt.tight_layout()
    # Save the figure
    plt.savefig(file_path, dpi=300, format='pdf')
    plt.show()

# Read the combined DataFrame again to ensure no missing values affect the plot
main_file_path = "/data/muscat_data/jaguir26/project1_ucsc_phd/climate_indices/combined_indices_daily_standardized.csv"
soil_file_path = "/data/muscat_data/jaguir26/project1_ucsc_phd/climate_indices/soil_moisture_daily_avg.csv"
ppt_file_path = "/data/muscat_data/jaguir26/project1_ucsc_phd/PPT.csv"

df_main = pd.read_csv(main_file_path)
df_soil = pd.read_csv(soil_file_path)
df_ppt = pd.read_csv(ppt_file_path)

df_soil.rename(columns={'time': 'Date', 'average_soil_moisture': 'soil'}, inplace=True)
df_ppt.rename(columns={'time': 'Date'}, inplace=True)

df_combined = pd.merge(df_main, df_soil[['Date', 'soil']], on='Date', how='left')
df_combined = pd.merge(df_combined, df_ppt[['Date', 'ppt']], on='Date', how='left')

# Convert 'Date' to datetime format for plotting
df_combined['Date'] = pd.to_datetime(df_combined['Date'])

# Define the columns of interest and their full titles
columns_of_interest = {
    'ppt': 'Precipitation by Prism (ppt)',
    'soil': 'Soil Moisture (soil)',
    'Solar Flux': 'Solar Flux (Solar Flux)',
    'Niño 1+2': 'Niño 1+2 (Niño 1+2)',
    'ONI': 'Oceanic Niño Index (ONI)',
    'WHWP': 'Western Hemisphere Warm Pool (WHWP)',
    'GMT': 'Global Mean Temperature (GMT)',
    'NOI': 'Northern Oscillation Index (NOI)',
    'AMO': 'Atlantic Multidecadal Oscillation (AMO)',
    'TSA': 'Tropical South Atlantic Index (TSA)',
    'TNA': 'Tropical North Atlantic Index (TNA)',
    'SOI': 'Southern Oscillation Index (SOI)'
}

# Plot each time series in subplots with different colors and save the figure
start_date = '1987-05-29'
end_date = '2022-12-25'
output_file_path_different_colors = "/data/muscat_data/jaguir26/project1_ucsc_phd/aux_time_series_different_colors.pdf"
plot_time_series_subplots(df_combined, columns_of_interest, start_date, end_date, output_file_path_different_colors, custom_colors)

# Plot each time series in subplots with a single friendly blue color and save the figure
output_file_path_single_color = "/data/muscat_data/jaguir26/project1_ucsc_phd/aux_time_series_single_color.pdf"
plot_time_series_subplots(df_combined, columns_of_interest, start_date, end_date, output_file_path_single_color, [friendly_blue])
